# SIRAP USA — RT-DETRv2-S Training and OpenVINO Inference

**Project context.** This notebook is the research and preparation workflow for SIRAP's USA license-plate recognition profile. It brings dataset review, training, evaluation, OCR, CPU inference, and optional live-camera work into one English-language notebook.

**Model and runtime.** The detector is **RT-DETRv2-S with a ResNet-18vd backbone**. Fine-tuning uses Transformers and PyTorch. The trained checkpoint is exported to OpenVINO IR; detector inference uses the native Python OpenVINO Runtime. The starting checkpoint, [merve/license-plates-rtdetrv2](https://huggingface.co/merve/license-plates-rtdetrv2), declares Apache-2.0. The dataset and other dependencies have separate license terms that must be reviewed independently.

**Recognition flow.** On a full image or video frame, the detector proposes plate boxes. SIRAP crops candidate plates and the configured OCR reader returns text and confidence. Validation and test sections measure these stages separately. The optional live-camera section uses the same USA detector and OCR pipeline; it does not trigger a gate, apply an authorization list, or send readings to the application's private storage.

**Data and results.** The default training source is Roboflow's European + USA License plates export. The grouped holdout is built from that same export, so it is an internal test rather than an external dataset. Without vehicle or session identifiers, grouping is limited to source-name families and exact-file hashes. No training or evaluation metrics are embedded here: they appear only after the corresponding run completes.

## Notebook contents

| Section | Purpose |
|---|---|
| 0. Install dependencies | Install Transformers/PyTorch, COCO tools, OpenVINO Runtime, Roboflow, OCR, and image utilities. |
| 1. Clone or refresh the repository | Prepare a clean project checkout and make its Python modules available. |
| 2. Set paths and model identity | Select the USA profile, checkpoint, image size, output folders, and compute settings. |
| 3. Inspect the USA profile | Check the country configuration and the expected data, evaluation, and model folders. |
| 4. Select and download the Roboflow dataset | Read the Roboflow Secret and download the European + USA License plates COCO export. |
| 5. Convert full-scene annotations to COCO (optional) | Build detector data from complete scenes annotated with the repository CSV template. |
| 6. Create the grouped holdout | Deduplicate images, form source groups, prepare train/validation/test splits, and audit leakage. |
| 7. Validate the COCO splits | Check annotation files, image paths, categories, and boxes without scoring the test set. |
| 8. Review training annotations | Plot sample images and plate boxes so annotation problems can be corrected before training. |
| 9. Recommended training protocol | Explain the 50-epoch ceiling, patience 8, validation-only model selection, and sealed final test. |
| 10. Set training and search parameters | Configure the three training seeds, early stopping, and optional Optuna budget. |
| 11. Run the Optuna search (optional) | Compare training settings on validation data only. |
| 12. Train and select the three-seed candidate | Fine-tune, export OpenVINO IR, compare candidates on validation, and record the selected model. |
| 13. Load the USA OpenVINO IR | Load the country-specific model or a clearly identified shared technical fallback. |
| 14. Run a smoke check | Confirm that OpenVINO inference runs; a blank image is not an accuracy measurement. |
| 15. Calibrate the confidence threshold | Choose the operating threshold from validation predictions and save the selection record. |
| 16. Run the optional live-camera workflow | **16.1** generate the local camera bridge; **16.2** configure frame reading and the USA pipeline; **16.3** test the connection; **16.4** run and summarize a live session. |
| 17. Check image robustness | Compare validation performance under documented synthetic image changes. |
| 18. Measure CPU inference latency | Measure detector-only OpenVINO latency on validation images. |
| 19. Evaluate OCR on plate crops | Measure OCR on labeled, already-cropped USA plates. |
| 20. Evaluate SIRAP's PlateReader (optional) | Inspect OCR output on a user-supplied crop CSV without claiming unvalidated USA accuracy. |
| 21. Evaluate full scenes (optional) | Measure detection plus OCR on a separate, labeled USA scene set. |
| 22. Evaluate the SIRAP pipeline (optional) | Measure the configured detector and OCR pipeline together on a disjoint scene set. |
| 23. Run the held-out detector test once | Score the frozen model on the grouped COCO test after model selection is complete. |
| 24. Review the deployment checklist | Check data coverage, threshold choice, latency, human review, and access-control risks. |
| 25. Publish the reviewed OpenVINO IR (optional) | Upload only the reviewed model files using a write-enabled GitHub Secret. |
| 26. Review generated files | Identify the audit, training, calibration, OCR, scene, and final-test reports. |

## How to use this notebook

Run sections 0–3 first in a new Colab session. Choose one data route in section 4 or 5, then create and audit the grouped split in sections 6–8. Train on a GPU. Use validation data for Optuna, seed comparison, and threshold selection. Record the selected checkpoint and freeze it before any final detector test.

| Task | Required preparation |
|---|---|
| Download the default Roboflow source | Add ROBOFLOW_API_KEY to Colab Secrets, then run sections 0–4. |
| Prepare full-scene COCO data | Add complete scenes and annotations to the repository template, then run sections 0–3 and section 5. |
| Train a USA detector | Run sections 0–10, complete the split audit, and enable only the required Optuna or final-training switch. |
| Calibrate the detector | Load the selected USA OpenVINO IR, record the model selection, freeze it, then run section 15 on validation. |
| Use a Cloudflare-published camera | Run the bridge on the camera PC, start the tunnel, add CAMARA_URL and CAMARA_TOKEN to Colab Secrets, then use sections 16.1–16.4. The live session is disabled by default. |
| Evaluate OCR or full scenes | Supply the matching labeled CSV and disjoint data described in sections 19–22. |
| Evaluate the final detector | Complete the model-selection record and threshold calibration, then run section 23 once. |

Training, evaluation, and live processing are separate operations. All long-running switches begin disabled. Live camera output may contain personal data: it is written to the temporary Colab workspace, is not sent to SIRAP storage, and should not be committed to the public repository.

## Workflow and execution order

The first run prepares the environment and repository, selects a dataset, and builds an audited COCO split. Training and tuning use train/validation data. After selecting and freezing a checkpoint, calibrate its threshold on validation; reserve the grouped test for one final evaluation.

The optional live-camera workflow belongs after the detector and OCR pipeline are available. It reads the camera URL and token from Colab Secrets, tests connectivity before loading a stream, and remains disabled during “Run all.” Its CSV, JSON summary, and optional annotated video are saved under /content/SIRAP_workspace/live_camera/, outside the cloned public repository. Delete those temporary outputs when they are no longer needed.

## 0. Install dependencies

This single cell replaces repeated installation cells. It installs the Transformers/PyTorch training tools, COCO metrics, OpenVINO Runtime, Roboflow, OCR, YAML, and image utilities. If Colab asks to restart, do so and then continue from the repository setup section.

In [ ]:
import subprocess
import sys

packages = [
    "transformers>=5.4,<6",
    "accelerate>=1.2,<2",
    "pycocotools>=2.0.8",
    "optuna>=4,<5",
    "openvino>=2025.4,<2027",
    "fast-plate-ocr==1.1.0",
    "roboflow==1.7.1",
    "opencv-python-headless==4.12.0.88",
    "matplotlib>=3.8",
    "pandas>=2.2,<4",
    "PyYAML>=6,<7",
    "requests>=2.31,<3",
]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *packages])
print("Dependencies installed. If Colab asks you to restart the runtime, restart it and then continue from the repository setup section.")

## 1. Clone or refresh the project repository

This cell reuses and fast-forward-updates a clean clone when possible. Otherwise, it makes a fresh clone in one of two Colab locations. It refuses to overwrite non-empty folders or a clone with uncommitted changes, then adds the repository to Python's import path.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPOSITORY_URL = "https://github.com/eduardoceballos666/SIRAP_USA_ECUADOR.git"
preferred = Path("/content/SIRAP_USA_ECUADOR")
candidates = [preferred, Path("/content/SIRAP_USA_ECUADOR_notebook")]

def prepare_clone(path):
    if (path / ".git").is_dir():
        status = subprocess.run(
            ["git", "-C", str(path), "status", "--porcelain"],
            check=True, capture_output=True, text=True,
        ).stdout.strip()
        if status:
            return False
        subprocess.run(
            ["git", "-C", str(path), "pull", "--ff-only", "origin", "main"],
            check=True,
        )
        return True
    if path.exists() and any(path.iterdir()):
        return False
    subprocess.run(["git", "clone", "--depth", "1", REPOSITORY_URL, str(path)], check=True)
    return True

REPO_ROOT = next((path for path in candidates if prepare_clone(path)), None)
if REPO_ROOT is None:
    raise RuntimeError(
        "No clean clone folder was available. Existing folders were left untouched. "
        "Check /content/SIRAP_USA_ECUADOR and /content/SIRAP_USA_ECUADOR_notebook."
    )
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT))
print("Repository:", REPO_ROOT)

## 2. Set paths, model identity, and compute information

These variables define the USA profile, starting checkpoint, image size, final model directory, metrics directory, and temporary training directory. The suggested batch size is 8 for a GPU with at least 16 GB, otherwise 4. Fine-tuning is intended for a Colab GPU; CPU training can be very slow. OpenVINO CPU inference is benchmarked later.

In [ ]:
import json
import math
import shutil
import time
import hashlib
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw
from pathlib import Path

COUNTRY = "usa"
MODEL_ID = "merve/license-plates-rtdetrv2"
IMAGE_SIZE = 640
DATA_DIR = None
MODEL_OUTPUT = REPO_ROOT / "models/usa/detector/rtdetrv2_s"
METRICS_DIR = REPO_ROOT / "results/metrics"
METRICS_DIR.mkdir(parents=True, exist_ok=True)
RUNS_ROOT = Path("/content/rtdetrv2_usa_runs")
RUNS_ROOT.mkdir(parents=True, exist_ok=True)

GPU_AVAILABLE = torch.cuda.is_available()
GPU_MEMORY_GB = (
    torch.cuda.get_device_properties(0).total_memory / 1024**3
    if GPU_AVAILABLE else 0.0
)
BATCH_SIZE = 8 if GPU_MEMORY_GB >= 16 else 4

print("Country:", COUNTRY)
print("Starting checkpoint:", MODEL_ID)
print("Training device:", torch.cuda.get_device_name(0) if GPU_AVAILABLE else "CPU")
print(f"GPU memory: {GPU_MEMORY_GB:.1f} GB | recommended initial batch size: {BATCH_SIZE}")
print("Dataset root: not selected yet")

## 3. Inspect the USA profile and expected directories

This exploratory cell reads the country YAML used by the application and prints the configured detector. It also checks whether USA dataset, evaluation, and model directories exist. Missing directories are informational at this point.

In [ ]:
import yaml
cfg=yaml.safe_load((REPO_ROOT/"config/countries/usa.yaml").read_text())
print(cfg["detector"])
for p in [REPO_ROOT/"datasets/usa",REPO_ROOT/"evaluation/usa",REPO_ROOT/"models/usa"]:
    print(p.relative_to(REPO_ROOT),"exists:",p.exists())


## 4. Select and download the Roboflow dataset

The default source is European + USA License plates, workspace plateblur, project european-usa-license-plates, version 1, exported as COCO Segmentation. RT-DETR uses the COCO boxes and maps the three plate labels to one license_plate class during split creation. Add ROBOFLOW_API_KEY to Colab Secrets and allow this notebook to read it. Set USE_ROBOFLOW to False for the manual full-scene route below. For an existing external COCO dataset, edit the path assigned in the code's else branch.

An earlier inspection of version 1 reported 13,143 training images, 386 validation images, and 106 test images. The current run reads actual counts from the downloaded annotations; these figures are dataset counts, not model results.

In [ ]:
from src.train import (
    RF_FORMAT,
    RF_PROJECT,
    RF_TITLE,
    RF_VERSION,
    RF_WORKSPACE,
    _annotation_file,
    download_roboflow_dataset,
)

DATA_DIR = None
USE_ROBOFLOW = True
SOURCE_DATASET_NAME = RF_TITLE
ROBOFLOW_VERSION_USED = RF_VERSION

print(
    f"Roboflow source: {RF_TITLE} | "
    f"{RF_WORKSPACE}/{RF_PROJECT} | version {RF_VERSION} | {RF_FORMAT}"
)

if USE_ROBOFLOW:
    # Add ROBOFLOW_API_KEY to Colab Secrets before running this cell.
    DATA_DIR = str(download_roboflow_dataset("/content/datasets"))
else:
    DATA_DIR = str(REPO_ROOT / "datasets/usa/detection_coco")
    SOURCE_DATASET_NAME = "Manually supplied USA COCO dataset"
    ROBOFLOW_VERSION_USED = None

if DATA_DIR:
    for split in ("train", "valid", "test"):
        try:
            print(split, _annotation_file(DATA_DIR, split))
        except FileNotFoundError as exc:
            print(exc)

## 5. Optional route: convert annotated full scenes to COCO

Use this route when you have complete vehicle or access-lane images with plate boxes annotated in the repository CSV template. The CSV needs one row per plate box; negative scenes have blank coordinates. Keep images from the same vehicle, camera sequence, or session in one source_group. Complete annotations.csv before enabling conversion. This route sets DATA_DIR to the generated COCO folder.

In [ ]:
import shutil
from scripts.usa_annotations_to_coco import convert_annotations

MANUAL_DATA_ROOT = REPO_ROOT / "datasets/usa/detection_coco"
SOURCE_IMAGES = MANUAL_DATA_ROOT / "source_images"
ANNOTATION_TEMPLATE = MANUAL_DATA_ROOT / "annotations_template.csv"
ANNOTATIONS_CSV = MANUAL_DATA_ROOT / "annotations.csv"
SOURCE_IMAGES.mkdir(parents=True, exist_ok=True)

if not ANNOTATIONS_CSV.exists() and ANNOTATION_TEMPLATE.is_file():
    shutil.copy2(ANNOTATION_TEMPLATE, ANNOTATIONS_CSV)
    print("Copied the template to", ANNOTATIONS_CSV, "; add one row per plate box before conversion.")

RUN_COCO_CONVERSION = False
if RUN_COCO_CONVERSION:
    conversion_summary = convert_annotations(
        SOURCE_IMAGES, ANNOTATIONS_CSV, MANUAL_DATA_ROOT, verify_hashes=True
    )
    DATA_DIR = str(MANUAL_DATA_ROOT)
    SOURCE_DATASET_NAME = "Manually annotated USA scenes"
    ROBOFLOW_VERSION_USED = None
    print(conversion_summary)
else:
    print("Keep RUN_COCO_CONVERSION set to False until full-scene images and annotations are ready.")

## 6. Create the larger grouped holdout and prevent leakage

This cell consolidates the original preprocessing workflow. It reads COCO exports for train, validation, and test; resolves image paths; computes SHA-256 hashes; and groups augmentation variants by source filename. The optional source_group_manifest.csv can merge source files from the same vehicle or session. Exact duplicate files are merged across splits. Exact duplicates with conflicting boxes are excluded and listed in an audit CSV.

The original validation and test groups are held out. Additional source groups are reserved from training until the held-out set contains at least 400 images. A new validation subset is taken from the remaining training groups. The cell checks that no source group or exact image crosses train, validation, and test, then writes one-class COCO files, split manifests, and a JSON summary. Without vehicle/session identifiers, the grouping guarantee is limited to source filename families and exact hashes.

In [ ]:
import csv
import hashlib
import json
import random
import shutil
from collections import Counter, defaultdict
from pathlib import Path

if not DATA_DIR:
    raise RuntimeError("Download the COCO dataset or set DATA_DIR first.")

SOURCE_ROOT = Path(DATA_DIR).resolve()
INDEPENDENT_ROOT = SOURCE_ROOT.parent / "european-usa-license-plates-independent-coco"
GROUP_MANIFEST = SOURCE_ROOT / "source_group_manifest.csv"
ADDITIONAL_TEST_GROUP_FRACTION = 0.15
VALIDATION_GROUP_FRACTION = 0.15
MINIMUM_TEST_IMAGES = 400
SPLIT_SEED = 42

if not SOURCE_ROOT.is_dir():
    raise FileNotFoundError(f"Dataset directory does not exist: {SOURCE_ROOT}")

def source_key_from_filename(filename):
    basename = Path(filename).name
    return basename.split(".rf.", 1)[0] if ".rf." in basename else Path(basename).stem

def resolve_source_image(source_folder, filename):
    basename = Path(filename).name
    candidates = [
        source_folder / filename,
        source_folder / basename,
        SOURCE_ROOT / filename,
        SOURCE_ROOT / basename,
    ]
    for candidate in candidates:
        if candidate.is_file():
            return candidate.resolve()
    matches = list(SOURCE_ROOT.rglob(basename))
    if matches:
        return matches[0].resolve()
    raise FileNotFoundError(f"Annotated image was not found: {filename}")

def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def annotation_signature(record):
    return sorted(
        tuple(round(float(value), 4) for value in annotation["bbox"])
        for annotation in record["annotations"]
    )

source_payloads = {}
source_records = {}
category_maps = {}
all_source_keys = set()

for split_name in ("train", "valid", "test"):
    annotation_path = _annotation_file(SOURCE_ROOT, split_name)
    payload = json.loads(annotation_path.read_text(encoding="utf-8"))
    source_payloads[split_name] = payload
    category_maps[split_name] = {
        int(category["id"]): category.get("name", str(category["id"]))
        for category in payload.get("categories", [])
    }
    annotations_by_image = defaultdict(list)
    for annotation in payload.get("annotations", []):
        image_id = int(annotation["image_id"])
        if "bbox" not in annotation or len(annotation["bbox"]) != 4:
            raise ValueError(f"Annotation without a valid COCO bbox in {annotation_path}")
        annotations_by_image[image_id].append(annotation)

    image_ids = {int(image["id"]) for image in payload.get("images", [])}
    orphan_ids = set(annotations_by_image) - image_ids
    if orphan_ids:
        raise ValueError(f"Annotations have no matching image in {split_name}: {sorted(orphan_ids)[:5]}")

    records = []
    for image_record in payload.get("images", []):
        source_key = source_key_from_filename(image_record["file_name"])
        all_source_keys.add(source_key)
        record = {
            "source_split": split_name,
            "source_folder": annotation_path.parent,
            "image": image_record,
            "annotations": annotations_by_image.get(int(image_record["id"]), []),
            "source_key": source_key,
        }
        image_path = resolve_source_image(annotation_path.parent, image_record["file_name"])
        record["source_path"] = image_path
        record["sha256"] = sha256_file(image_path)
        records.append(record)
    source_records[split_name] = records

# Create or extend the manifest so it remains useful after a new dataset export.
if not GROUP_MANIFEST.is_file():
    with GROUP_MANIFEST.open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=["source_key", "group_id"])
        writer.writeheader()
        writer.writerows({"source_key": key, "group_id": ""} for key in sorted(all_source_keys))
    print("Group manifest template created:", GROUP_MANIFEST)

with GROUP_MANIFEST.open(newline="", encoding="utf-8") as handle:
    reader = csv.DictReader(handle)
    if not {"source_key", "group_id"}.issubset(reader.fieldnames or []):
        raise ValueError("The manifest must contain source_key and group_id columns.")
    manifest_rows = list(reader)

manifest_by_key = {}
for row in manifest_rows:
    key = (row.get("source_key") or "").strip()
    group_id = (row.get("group_id") or "").strip()
    if not key:
        continue
    if key in manifest_by_key and manifest_by_key[key] != group_id:
        raise ValueError(f"Duplicate source_key assigned to different groups: {key}")
    manifest_by_key[key] = group_id

missing_keys = sorted(all_source_keys - set(manifest_by_key))
if missing_keys:
    with GROUP_MANIFEST.open("a", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=["source_key", "group_id"])
        writer.writerows({"source_key": key, "group_id": ""} for key in missing_keys)
    print(f"Added {len(missing_keys)} new keys to the template.")

for records in source_records.values():
    for record in records:
        manual_group = manifest_by_key.get(record["source_key"], "")
        record["base_group_id"] = (
            f"manual:{manual_group}" if manual_group else f"source:{record['source_key']}"
        )

# The same exact file must never appear in different splits.
parent = {
    record["base_group_id"]: record["base_group_id"]
    for records in source_records.values()
    for record in records
}

def find_group(group_id):
    while parent[group_id] != group_id:
        parent[group_id] = parent[parent[group_id]]
        group_id = parent[group_id]
    return group_id

def union_groups(left, right):
    root_left, root_right = find_group(left), find_group(right)
    if root_left == root_right:
        return
    first, second = sorted((root_left, root_right))
    parent[second] = first

records_by_hash = defaultdict(list)
for records in source_records.values():
    for record in records:
        records_by_hash[record["sha256"]].append(record)

conflicting_sha256 = set()
ambiguous_duplicate_audit_rows = []
for digest, copies in records_by_hash.items():
    signatures = {tuple(annotation_signature(record)) for record in copies}
    if len(signatures) > 1:
        conflicting_sha256.add(digest)
        for record in copies:
            ambiguous_duplicate_audit_rows.append({
                "sha256": digest,
                "source_split": record["source_split"],
                "file_name": record["image"]["file_name"],
                "source_key": record["source_key"],
                "annotation_bboxes": json.dumps(
                    [annotation["bbox"] for annotation in record["annotations"]],
                    ensure_ascii=False,
                ),
            })
    group_ids = sorted({record["base_group_id"] for record in copies})
    for group_id in group_ids[1:]:
        union_groups(group_ids[0], group_id)

for records in source_records.values():
    for record in records:
        record["group_id"] = f"group:{find_group(record['base_group_id'])}"

ambiguous_duplicate_records_removed = len(ambiguous_duplicate_audit_rows)
for split_name, records in source_records.items():
    source_records[split_name] = [
        record for record in records if record["sha256"] not in conflicting_sha256
    ]
print(
    "Exact duplicate images excluded due to conflicting bounding boxes:",
    ambiguous_duplicate_records_removed,
    "| hashes conflictivos:",
    len(conflicting_sha256),
)

rng = random.Random(SPLIT_SEED)

def one_record_per_group(records):
    grouped = defaultdict(list)
    for record in records:
        grouped[record["group_id"]].append(record)
    selected = []
    for group_id in sorted(grouped):
        choices = sorted(grouped[group_id], key=lambda item: item["image"]["file_name"])
        selected.append(rng.choice(choices))
    return selected

# The original validation and test splits are both held out for final testing.
original_heldout = source_records["valid"] + source_records["test"]
test_base_records = one_record_per_group(original_heldout)
original_heldout_groups = {record["group_id"] for record in original_heldout}

train_by_group = defaultdict(list)
for record in source_records["train"]:
    if record["group_id"] not in original_heldout_groups:
        train_by_group[record["group_id"]].append(record)

available_train_groups = sorted(train_by_group)
if len(available_train_groups) < 3:
    raise ValueError("Fewer than three groups remain to create train, validation, and test splits.")

rng.shuffle(available_train_groups)
desired_extra_test_groups = max(
    round(len(available_train_groups) * ADDITIONAL_TEST_GROUP_FRACTION),
    max(0, MINIMUM_TEST_IMAGES - len(test_base_records)),
)
extra_test_group_count = min(desired_extra_test_groups, len(available_train_groups) - 2)
if len(test_base_records) + extra_test_group_count < MINIMUM_TEST_IMAGES:
    raise ValueError(
        "There are not enough independent groups to create a test split of "
        f"{MINIMUM_TEST_IMAGES} images."
    )

extra_test_group_ids = set(available_train_groups[:extra_test_group_count])
remaining_train_group_ids = available_train_groups[extra_test_group_count:]
extra_test_records = one_record_per_group([
    record
    for group_id in extra_test_group_ids
    for record in train_by_group[group_id]
])

validation_group_count = max(
    1, round(len(remaining_train_group_ids) * VALIDATION_GROUP_FRACTION)
)
validation_group_count = min(validation_group_count, len(remaining_train_group_ids) - 1)
validation_group_ids = set(remaining_train_group_ids[:validation_group_count])

validation_records = one_record_per_group([
    record
    for group_id in validation_group_ids
    for record in train_by_group[group_id]
])
training_records = [
    record
    for group_id in remaining_train_group_ids
    if group_id not in validation_group_ids
    for record in train_by_group[group_id]
]
test_records = test_base_records + extra_test_records

# Remove exact duplicates from train so they are not over-weighted.
seen_train_hashes = set()
deduplicated_training = []
for record in sorted(training_records, key=lambda item: item["image"]["file_name"]):
    if record["sha256"] not in seen_train_hashes:
        seen_train_hashes.add(record["sha256"])
        deduplicated_training.append(record)
duplicate_train_images_removed = len(training_records) - len(deduplicated_training)
training_records = deduplicated_training

split_records = {
    "train": training_records,
    "valid": validation_records,
    "test": test_records,
}
group_sets = {
    name: {record["group_id"] for record in records}
    for name, records in split_records.items()
}
hash_sets = {
    name: {record["sha256"] for record in records}
    for name, records in split_records.items()
}
for left, right in (("train", "valid"), ("train", "test"), ("valid", "test")):
    group_overlap = group_sets[left] & group_sets[right]
    hash_overlap = hash_sets[left] & hash_sets[right]
    if group_overlap or hash_overlap:
        raise RuntimeError(
            f"Leakage between {left} y {right}: "
            f"{len(group_overlap)} groups and {len(hash_overlap)} hashes."
        )

if len(test_records) < MINIMUM_TEST_IMAGES:
    raise RuntimeError(f"The final test split contains only {len(test_records)} images.")

marker = INDEPENDENT_ROOT / ".sirap_independent_split.json"
if INDEPENDENT_ROOT.exists():
    if not marker.is_file():
        raise FileExistsError(
            f"{INDEPENDENT_ROOT} exists and was not created by this notebook; "
            "it will not be overwritten."
        )
    shutil.rmtree(INDEPENDENT_ROOT)
INDEPENDENT_ROOT.mkdir(parents=True)

source_licenses = next(
    (payload.get("licenses", []) for payload in source_payloads.values()
     if payload.get("licenses")),
    [],
)
manifest_rows = []
split_summary = {}

for split_name, records in split_records.items():
    split_folder = INDEPENDENT_ROOT / split_name
    split_folder.mkdir(parents=True, exist_ok=True)
    output_images = []
    output_annotations = []
    source_category_counts = Counter()
    annotation_id = 0

    for image_id, record in enumerate(
        sorted(records, key=lambda item: (item["source_split"], item["image"]["file_name"])),
        start=1,
    ):
        original_name = Path(record["image"]["file_name"]).name
        output_name = f"{record['source_split']}__{record['sha256'][:12]}__{original_name}"
        destination = split_folder / output_name
        try:
            destination.symlink_to(record["source_path"])
        except (OSError, NotImplementedError):
            shutil.copy2(record["source_path"], destination)

        image_out = dict(record["image"])
        image_out.update({
            "id": image_id,
            "file_name": output_name,
            "sirap_source_key": record["source_key"],
            "sirap_group_id": record["group_id"],
            "sirap_sha256": record["sha256"],
            "sirap_original_split": record["source_split"],
        })
        output_images.append(image_out)

        category_map = category_maps[record["source_split"]]
        for source_annotation in record["annotations"]:
            bbox = [float(value) for value in source_annotation["bbox"]]
            x, y, width, height = bbox
            if x < 0 or y < 0 or width <= 0 or height <= 0:
                raise ValueError(f"Invalid COCO bbox in {original_name}: {bbox}")
            annotation_id += 1
            output_annotations.append({
                "id": annotation_id,
                "image_id": image_id,
                "category_id": 1,
                "bbox": bbox,
                "area": float(source_annotation.get("area", width * height)),
                "iscrowd": int(source_annotation.get("iscrowd", 0)),
            })
            source_category_counts[
                category_map.get(int(source_annotation["category_id"]), "unknown")
            ] += 1

        manifest_rows.append({
            "split": split_name,
            "file_name": output_name,
            "source_file_name": record["image"]["file_name"],
            "source_key": record["source_key"],
            "group_id": record["group_id"],
            "sha256": record["sha256"],
            "original_split": record["source_split"],
            "boxes": len(record["annotations"]),
        })

    empty_images = sum(
        1 for image in output_images
        if not any(annotation["image_id"] == image["id"] for annotation in output_annotations)
    )
    split_summary[split_name] = {
        "images": len(output_images),
        "groups": len({record["group_id"] for record in records}),
        "plates": len(output_annotations),
        "empty_images": empty_images,
        "source_category_annotations": dict(sorted(source_category_counts.items())),
    }
    coco_output = {
        "info": {
            "description": "SIRAP USA grouped internal test for RT-DETRv2-S",
            "source_dataset": SOURCE_DATASET_NAME,
            "grouping": "manual vehicle/session id when supplied; otherwise source filename and exact SHA-256",
        },
        "licenses": source_licenses,
        "images": output_images,
        "annotations": output_annotations,
        "categories": [{"id": 1, "name": "license_plate", "supercategory": "license_plate"}],
    }
    (split_folder / "_annotations.coco.json").write_text(
        json.dumps(coco_output, indent=2, ensure_ascii=False), encoding="utf-8"
    )

with (INDEPENDENT_ROOT / "excluded_ambiguous_exact_duplicates.csv").open(
    "w", newline="", encoding="utf-8"
) as handle:
    writer = csv.DictWriter(
        handle,
        fieldnames=["sha256", "source_split", "file_name", "source_key", "annotation_bboxes"],
    )
    writer.writeheader()
    writer.writerows(ambiguous_duplicate_audit_rows)

with (INDEPENDENT_ROOT / "split_manifest.csv").open(
    "w", newline="", encoding="utf-8"
) as handle:
    writer = csv.DictWriter(
        handle,
        fieldnames=[
            "split", "file_name", "source_file_name", "source_key",
            "group_id", "sha256", "original_split", "boxes",
        ],
    )
    writer.writeheader()
    writer.writerows(manifest_rows)

train_group_overlap_with_original_holdout = {
    record["group_id"]
    for record in source_records["train"]
    if record["group_id"] in original_heldout_groups
}
SPLIT_SUMMARY = {
    "source_dataset": SOURCE_DATASET_NAME,
    "roboflow_version": ROBOFLOW_VERSION_USED,
    "seed": SPLIT_SEED,
    "minimum_test_images": MINIMUM_TEST_IMAGES,
    "additional_test_group_fraction": ADDITIONAL_TEST_GROUP_FRACTION,
    "validation_group_fraction": VALIDATION_GROUP_FRACTION,
    "test_design": (
        "one representative per group from original valid+test, plus one "
        "representative per randomly reserved train source group"
    ),
    "grouping_level": (
        "manual vehicle/session ids where supplied, otherwise source filename "
        "family; exact SHA-256 duplicates are merged"
    ),
    "manual_source_keys_grouped": sum(bool(value) for value in manifest_by_key.values()),
    "source_images": {name: len(records) for name, records in source_records.items()},
    "train_groups_removed_for_original_valid_test_overlap": len(train_group_overlap_with_original_holdout),
    "additional_train_groups_reserved_for_test": extra_test_group_count,
    "validation_groups_reserved_from_remaining_train": validation_group_count,
    "exact_duplicate_train_images_removed": duplicate_train_images_removed,
    "ambiguous_exact_duplicate_hashes_excluded": len(conflicting_sha256),
    "ambiguous_exact_duplicate_records_excluded": ambiguous_duplicate_records_removed,
    "ambiguous_duplicate_audit": str(INDEPENDENT_ROOT / "excluded_ambiguous_exact_duplicates.csv"),
    "unique_sha256_audited": len({record["sha256"] for records in split_records.values() for record in records}),
    "splits": split_summary,
}
(INDEPENDENT_ROOT / "split_summary.json").write_text(
    json.dumps(SPLIT_SUMMARY, indent=2, ensure_ascii=False), encoding="utf-8"
)
marker.write_text(
    json.dumps({"generated_by": "SIRAP USA grouped COCO split", "seed": SPLIT_SEED}, indent=2),
    encoding="utf-8",
)

DATA_DIR = str(INDEPENDENT_ROOT)
INDEPENDENT_TEST_READY = True
print(json.dumps(SPLIT_SUMMARY, indent=2, ensure_ascii=False))
print("New DATA_DIR for training and evaluation:", DATA_DIR)
print("Optional vehicle/session manifest:", GROUP_MANIFEST)

## 7. Validate the resulting COCO splits

This audit loads each split through SIRAP's COCO dataset class and checks annotation files, category count, image paths, and bounding-box dimensions. DATA_READY is true only when train and validation are usable and contain exactly one class. TEST_READY records whether the held-out split is available; this cell does not score the test data.

In [ ]:
from src.train import CocoPlateDataset, _annotation_file

SPLITS = {}
DATA_READY = False
TEST_READY = False

if not DATA_DIR.is_dir():
    print("The COCO scene-data folder is not configured.")
    print("Set DATA_DIR and rerun this cell. OCR crop folders do not replace full-scene images with bounding boxes.")
else:
    for split in ("train", "val", "test"):
        try:
            dataset = CocoPlateDataset(DATA_DIR, split, image_processor=None)
            payload = json.loads(dataset.annotation_path.read_text(encoding="utf-8"))
            categories = payload.get("categories", [])
            missing_images, invalid_boxes = [], []
            for record in dataset.images:
                try:
                    image_path = dataset._image_path(record["file_name"])
                except FileNotFoundError:
                    missing_images.append(record["file_name"])
                    continue
                for ann in dataset.annotations_by_image.get(int(record["id"]), []):
                    x, y, w, h = map(float, ann["bbox"])
                    if w <= 0 or h <= 0 or x < 0 or y < 0:
                        invalid_boxes.append((record["file_name"], ann["bbox"]))
            SPLITS[split] = {
                "dataset": dataset,
                "payload": payload,
                "categories": categories,
                "images": len(dataset.images),
                "boxes": sum(len(v) for v in dataset.annotations_by_image.values()),
                "missing_images": missing_images,
                "invalid_boxes": invalid_boxes,
            }
            print(
                f"{split}: {len(dataset.images)} images, "
                f"{SPLITS[split]['boxes']} boxes, "
                f"{len(categories)} categories, "
                f"{len(missing_images)} missing images, "
                f"{len(invalid_boxes)} invalid boxes"
            )
        except FileNotFoundError:
            print(f"No COCO annotation was found for {split}; train and valid are required, and test is required for final evaluation.")

    DATA_READY = (
        "train" in SPLITS and "val" in SPLITS
        and len(SPLITS["train"]["categories"]) == 1
        and len(SPLITS["val"]["categories"]) == 1
        and not SPLITS["train"]["missing_images"]
        and not SPLITS["val"]["missing_images"]
    )
    TEST_READY = "test" in SPLITS and not SPLITS["test"]["missing_images"]
    print("Data ready for training:", DATA_READY)
    print("Test split available:", TEST_READY)
    if not DATA_READY:
        print("Check that there is one plate category and that COCO files and image paths are valid before training.")

## 8. Visually inspect training annotations

The plot displays up to four training images with their COCO plate boxes in green. Inspect small plates, partial occlusions, glare, boxes that include too much background, and negative images with no boxes. If you find issues, correct the source annotations and rebuild the COCO splits before training.

In [ ]:
from src.data_processing import load_image

if DATA_READY:
    sample_data = SPLITS["train"]["dataset"]
    sample_count = min(4, len(sample_data.images))
    fig, axes = plt.subplots(1, sample_count, figsize=(5 * sample_count, 4))
    if sample_count == 1:
        axes = [axes]
    for ax, record in zip(axes, sample_data.images[:sample_count]):
        image_path = sample_data._image_path(record["file_name"])
        image = Image.fromarray(load_image(image_path.read_bytes()))
        draw = ImageDraw.Draw(image)
        for ann in sample_data.annotations_by_image.get(int(record["id"]), []):
            x, y, w, h = map(float, ann["bbox"])
            draw.rectangle((x, y, x + w, y + h), outline="lime", width=3)
        ax.imshow(image)
        ax.set_title(record["file_name"])
        ax.axis("off")
    plt.tight_layout()
else:
    print("Configure and validate DATA_DIR before reviewing examples.")

## 9. Recommended training protocol

The USA recipe uses a maximum of **50 epochs**, early-stopping patience **8**, a starting learning rate of 5e-5, weight decay 1e-4, and image size 640. Validation is evaluated once per epoch. The trainer retains the checkpoint with the lowest validation loss; the notebook then compares three independently trained seeds by validation mAP50-95 and copies the winner into the USA model directory.

Fifty epochs are a ceiling, not a requirement to complete every epoch. Patience 8 means training can stop after eight consecutive validation checks without improvement. Optuna is an optional screening stage with at most 12 epochs per trial; it uses validation mAP50-95 only. The detector test remains sealed until the checkpoint and threshold are frozen.

Use a GPU for fine-tuning. If memory is limited, start with batch size 4; do not alter the test set to accommodate training constraints.

## 10. Set training, search, and seed parameters

This is the single source of truth for the USA recipe. Three fixed seeds support a repeatability check; the notebook selects one model by validation mAP50-95 rather than averaging weights. MAX_TRIALS limits the Optuna budget. Record any changes in the final model-selection note.

After the three-seed comparison, fill in MODEL_SELECTION_RECORD with the selected checkpoint and validation evidence. Run threshold calibration next; only after the checkpoint and operational threshold are fixed should you set MODEL_CHECKPOINT_FROZEN to True.

In [ ]:
EPOCHS_FINAL = 50
EARLY_STOPPING_PATIENCE = 8
MAX_TRIALS = 8
EPOCHS_TUNING = 12
FINAL_SEEDS = [0, 1, 2]
MODEL_CHECKPOINT_FROZEN = False
MODEL_SELECTION_RECORD = ""
DEFAULT_PARAMS = {
    "learning_rate": 5e-5,
    "weight_decay": 1e-4,
    "batch_size": BATCH_SIZE,
}
SELECTED_PARAMS = dict(DEFAULT_PARAMS)
print(SELECTED_PARAMS)

## 11. Optional Optuna search

This search compares learning rate, weight decay, and batch size using validation mAP50-95. It trains a short run for each trial, exports each candidate to OpenVINO IR, evaluates only on validation, and saves the best settings. Leave RUN_OPTUNA set to False until the dataset passes the COCO audit and a GPU budget is available. Never tune against the test split.

In [ ]:
from src.train import train_detector
from src.evaluate import evaluate_detector
import optuna

RUN_OPTUNA = False
TUNING_DIR = RUNS_ROOT / "optuna"

def objective(trial):
    learning_rate = trial.suggest_float("learning_rate", 1e-5, 1e-4, log=True)
    weight_decay = trial.suggest_float("weight_decay", 1e-5, 5e-4, log=True)
    batch_choices = [4, 8] if GPU_MEMORY_GB >= 16 else [4]
    batch_size = trial.suggest_categorical("batch_size", batch_choices)
    output_dir = TUNING_DIR / f"trial_{trial.number}"
    train_detector(
        DATA_DIR, country=COUNTRY, model_id=MODEL_ID,
        output_dir=output_dir, run_dir=output_dir / "trainer",
        epochs=EPOCHS_TUNING, early_stopping_patience=EARLY_STOPPING_PATIENCE,
        batch_size=batch_size,
        learning_rate=learning_rate, weight_decay=weight_decay,
        seed=42, force=True,
    )
    metrics = evaluate_detector(
        DATA_DIR, country=COUNTRY, split="val",
        checkpoint=output_dir / "openvino", model_id=MODEL_ID,
        threshold=0.001,
        result_path=METRICS_DIR / f"optuna_usa_trial_{trial.number}.json",
    )
    score = metrics.get("map50_95")
    if score is None:
        raise RuntimeError("Validation did not produce mAP50-95.")
    return float(score)

if RUN_OPTUNA:
    if not DATA_READY:
        raise RuntimeError("Fix DATA_DIR before starting Optuna.")
    study = optuna.create_study(direction="maximize", study_name="rtdetrv2_s_usa")
    study.optimize(objective, n_trials=MAX_TRIALS)
    study.trials_dataframe().to_csv(METRICS_DIR / "rtdetr_usa_optuna_trials.csv", index=False)
    optuna_summary = {
        "country": COUNTRY,
        "model_id": MODEL_ID,
        "objective": "map50_95_validation",
        "best_value": study.best_value,
        "best_params": study.best_trial.params,
        "trial_count": len(study.trials),
        "maximum_epochs_per_trial": EPOCHS_TUNING,
        "detector": "RT-DETRv2-S",
        "inference_runtime": "OpenVINO Runtime",
    }
    (METRICS_DIR / "rtdetr_usa_optuna_summary.json").write_text(
        json.dumps(optuna_summary, indent=2), encoding="utf-8"
    )
    SELECTED_PARAMS = {
        "learning_rate": study.best_params["learning_rate"],
        "weight_decay": study.best_params["weight_decay"],
        "batch_size": study.best_params["batch_size"],
    }
    print("Best validation configuration:", SELECTED_PARAMS)
else:
    print("Optuna is disabled. The initial values will be used until the search is run.")

## 12. Train and select the final three-seed candidate

For each seed, train_detector fine-tunes the starting checkpoint with Transformers/PyTorch, evaluates once per epoch, applies early stopping, and exports both the Transformers checkpoint and OpenVINO IR. The notebook ranks the three candidates by validation mAP50-95 and copies the winner into models/usa/detector/rtdetrv2_s/. It records the candidate metrics as well. Leave the run disabled until data, labels, and training settings are ready.

In [ ]:
RUN_FINAL_TRAINING = False

if RUN_FINAL_TRAINING:
    MODEL_CHECKPOINT_FROZEN = False
    MODEL_SELECTION_RECORD = ""
    old_threshold_path = METRICS_DIR / "rtdetr_usa_threshold_calibration.json"
    if old_threshold_path.is_file():
        from datetime import datetime, timezone
        archived_threshold_path = METRICS_DIR / (
            f"rtdetr_usa_threshold_calibration_previous_"
            f"{datetime.now(timezone.utc):%Y%m%d_%H%M%S}.json"
        )
        old_threshold_path.rename(archived_threshold_path)
        print("Previous threshold calibration archived at:", archived_threshold_path)
    if not DATA_READY:
        raise RuntimeError("Training will not start: a valid COCO dataset is missing.")
    candidates = []
    for seed in FINAL_SEEDS:
        run_dir = RUNS_ROOT / f"seed_{seed}"
        train_detector(
            DATA_DIR, country=COUNTRY, model_id=MODEL_ID,
            output_dir=run_dir, run_dir=run_dir / "trainer",
            epochs=EPOCHS_FINAL,
            early_stopping_patience=EARLY_STOPPING_PATIENCE,
            batch_size=SELECTED_PARAMS["batch_size"],
            learning_rate=SELECTED_PARAMS["learning_rate"],
            weight_decay=SELECTED_PARAMS["weight_decay"],
            seed=seed, force=True,
        )
        metrics = evaluate_detector(
            DATA_DIR, country=COUNTRY, split="val",
            checkpoint=run_dir / "openvino", model_id=MODEL_ID,
            threshold=0.001,
            result_path=METRICS_DIR / f"rtdetr_usa_val_seed_{seed}.json",
        )
        candidates.append({
            "seed": seed,
            "run_dir": str(run_dir),
            "map50_95": metrics.get("map50_95"),
            "map50": metrics.get("map50"),
            "recall": metrics.get("recall"),
            "precision": metrics.get("precision"),
        })

    ranked = [item for item in candidates if item["map50_95"] is not None]
    if not ranked:
        raise RuntimeError("No candidate produced mAP50-95.")
    best_run = max(ranked, key=lambda item: item["map50_95"])
    source_dir = Path(best_run["run_dir"])
    MODEL_OUTPUT.mkdir(parents=True, exist_ok=True)

    if any(MODEL_OUTPUT.iterdir()):
        from datetime import datetime, timezone
        backup_dir = Path("/content") / f"usa_rtdetr_backup_{datetime.now(timezone.utc):%Y%m%d_%H%M%S}"
        shutil.copytree(MODEL_OUTPUT, backup_dir, dirs_exist_ok=True)
        print("Previous model files were backed up to:", backup_dir)

    shutil.copytree(source_dir / "transformers", MODEL_OUTPUT / "transformers", dirs_exist_ok=True)
    shutil.copytree(source_dir / "openvino", MODEL_OUTPUT / "openvino", dirs_exist_ok=True)
    (METRICS_DIR / "rtdetr_usa_three_seed_validation.json").write_text(
        json.dumps({"candidates": candidates, "selected": best_run}, indent=2),
        encoding="utf-8",
    )
    validation_csv = METRICS_DIR / "rtdetr_usa_three_seed_validation.csv"
    pd.DataFrame(candidates).to_csv(validation_csv, index=False)
    selected_manifest = MODEL_OUTPUT / "openvino" / "sirap_training.json"
    english_manifest = METRICS_DIR / "rtdetr_usa_training_manifest.json"
    if selected_manifest.is_file():
        shutil.copy2(selected_manifest, english_manifest)
    print("USA model selected using validation:", best_run)
    print("Three-seed validation results:", validation_csv)
    print("Selected training manifest:", english_manifest if english_manifest.is_file() else "not found")
    print("OpenVINO IR:", MODEL_OUTPUT / "openvino" / "openvino_model.xml")
else:
    print("Final training is disabled. Set RUN_FINAL_TRAINING to True to start the three runs.")

## 13. Load the USA OpenVINO IR

The country-specific IR is preferred. If it is absent, this cell exports or loads the shared checkpoint for a smoke check; that shared model is only a technical fallback and does not establish USA performance. The detector is loaded on CPU through the native OpenVINO API. Fine-tuning must happen before this inference step.

In [ ]:
from src.export_openvino import export_openvino_model
from src.model import PlateDetector

COUNTRY_IR_DIR = MODEL_OUTPUT / "openvino"
SHARED_IR_DIR = REPO_ROOT / "models/shared/detector/rtdetrv2_s/openvino"
COUNTRY_IR_XML = COUNTRY_IR_DIR / "openvino_model.xml"

if COUNTRY_IR_XML.is_file():
    DETECTOR_DIR = COUNTRY_IR_DIR
    MODEL_IS_COUNTRY_FINETUNED = True
else:
    DETECTOR_DIR = SHARED_IR_DIR
    MODEL_IS_COUNTRY_FINETUNED = False
    if not (SHARED_IR_DIR / "openvino_model.xml").is_file():
        export_openvino_model(MODEL_ID, SHARED_IR_DIR, image_size=IMAGE_SIZE)

detector = PlateDetector(
    checkpoint=DETECTOR_DIR,
    model_id=MODEL_ID,
    imgsz=IMAGE_SIZE,
    conf=0.001,
    class_id=0,
    device="CPU",
    fallback_checkpoint=None,
)
print("Loaded model:", detector.name)
print("Fine-tuned specifically for the USA:", MODEL_IS_COUNTRY_FINETUNED)
print("OpenVINO devices:", detector.core.available_devices)

## 14. Run a quick OpenVINO smoke check

A blank image is sent through the detector to confirm that the IR loads and inference returns without an exception. The number of boxes on a blank image is not an accuracy metric and must not be reported as model performance.

In [ ]:
blank_image = np.zeros((480, 640, 3), dtype=np.uint8)
smoke_detections = detector.detect(blank_image, max_detections=20)
print("OpenVINO inference completed.")
print("Boxes returned for the blank image:", len(smoke_detections))
print("This check only confirms that the model loads and runs; it does not measure accuracy.")

## 15. Calibrate the confidence threshold on validation

This cell reuses validation detections to compare confidence thresholds from 0.05 to 0.95 at IoU 0.50. It selects the highest-recall threshold among those meeting the target precision of 0.99. If none reaches that precision, it reports the best F1 as a provisional reference and records that the target was missed. This is a validation decision, not a test result. The default threshold is 0.25 until calibration is run.

In [ ]:
def box_iou_xyxy(a, b):
    ix1, iy1 = max(a[0], b[0]), max(a[1], b[1])
    ix2, iy2 = min(a[2], b[2]), min(a[3], b[3])
    inter = max(0.0, ix2 - ix1) * max(0.0, iy2 - iy1)
    area_a = max(0.0, a[2] - a[0]) * max(0.0, a[3] - a[1])
    area_b = max(0.0, b[2] - b[0]) * max(0.0, b[3] - b[1])
    return inter / max(area_a + area_b - inter, 1e-9)

def collect_detection_records(split, max_images=None, transform=None):
    dataset = CocoPlateDataset(DATA_DIR, split, image_processor=None)
    source_images = dataset.images if max_images is None else dataset.images[:max_images]
    records = []
    for item in source_images:
        image_path = dataset._image_path(item["file_name"])
        image = load_image(image_path.read_bytes())
        if transform is not None:
            image = transform(image)
        gt_boxes = []
        for ann in dataset.annotations_by_image.get(int(item["id"]), []):
            x, y, w, h = map(float, ann["bbox"])
            gt_boxes.append((x, y, x + w, y + h))
        detections = detector.detect(image, max_detections=300)
        records.append({
            "image_id": int(item["id"]),
            "predictions": [(d.box, d.confidence) for d in detections],
            "ground_truth": gt_boxes,
        })
    return records

def score_detection_records(records, threshold, iou_threshold=0.50):
    tp = fp = fn = 0
    for record in records:
        unmatched = set(range(len(record["ground_truth"])))
        predictions = sorted(
            (item for item in record["predictions"] if item[1] >= threshold),
            key=lambda item: item[1], reverse=True,
        )
        for box, _ in predictions:
            best_index, best_iou = None, 0.0
            for index in unmatched:
                value = box_iou_xyxy(box, record["ground_truth"][index])
                if value > best_iou:
                    best_index, best_iou = index, value
            if best_index is not None and best_iou >= iou_threshold:
                tp += 1
                unmatched.remove(best_index)
            else:
                fp += 1
        fn += len(unmatched)
    precision = tp / max(tp + fp, 1)
    recall = tp / max(tp + fn, 1)
    f1 = 2 * precision * recall / max(precision + recall, 1e-12)
    return {"tp": tp, "fp": fp, "fn": fn, "precision": precision, "recall": recall, "f1": f1}

RUN_THRESHOLD_CALIBRATION = False
TARGET_PRECISION = 0.99
ACTIVE_THRESHOLD = 0.25
if RUN_THRESHOLD_CALIBRATION:
    if not DATA_READY or not COUNTRY_IR_XML.is_file():
        raise RuntimeError("Training data, validation data, and a USA fine-tuned IR are required for calibration.")
    if not MODEL_SELECTION_RECORD.strip():
        raise RuntimeError("Record the validation-selected checkpoint before calibrating its threshold.")
    val_records = collect_detection_records("val")
    threshold_rows = [
        {"threshold": round(float(value), 2), **score_detection_records(val_records, float(value))}
        for value in np.arange(0.05, 1.00, 0.05)
    ]
    threshold_table = pd.DataFrame(threshold_rows)
    eligible = threshold_table[threshold_table["precision"] >= TARGET_PRECISION]
    if len(eligible):
        selected = eligible.sort_values(["recall", "threshold"], ascending=[False, True]).iloc[0]
        selection_rule = "highest recall subject to the target precision"
    else:
        selected = threshold_table.sort_values("f1", ascending=False).iloc[0]
        selection_rule = "best F1; the target precision was not reached"
    selected_threshold = float(selected["threshold"])
    ACTIVE_THRESHOLD = selected_threshold
    result = {
        "country": COUNTRY,
        "model": "RT-DETRv2-S",
        "runtime": "OpenVINO Runtime Python",
        "split": "valid",
        "checkpoint": str(COUNTRY_IR_DIR),
        "model_selection_record": MODEL_SELECTION_RECORD,
        "iou_threshold": 0.50,
        "target_precision": TARGET_PRECISION,
        "selection_rule": selection_rule,
        "threshold": selected_threshold,
        "metrics": {key: float(selected[key]) for key in ("precision", "recall", "f1")},
    }
    (METRICS_DIR / "rtdetr_usa_threshold_calibration.json").write_text(
        json.dumps(result, indent=2), encoding="utf-8"
    )
    display(threshold_table)
    print("Validation threshold candidate:", result)
else:
    print("Calibration is disabled. Set RUN_THRESHOLD_CALIBRATION to True after training.")

## 16. Optional live mode: Cloudflare-published camera

This section adapts the former live-camera experiment to the current USA pipeline: RT-DETRv2-S detection through native OpenVINO Runtime, followed by the configured OCR reader. A camera in a private plant network is not directly reachable from a Colab runtime, so the bridge runs next to the camera and Cloudflare Tunnel carries HTTPS requests to it.

The mode is optional. It requires the bridge process, a tunnel, and the Colab Secrets described below. No camera connection is opened until the connection-check or live-session cell runs. Live processing is disabled by default. The four subsections create the bridge script, define a latest-frame reader and USA pipeline, check the connection, then run a bounded session with local outputs. This notebook does not track vehicles, decide access, open a barrier, or send live readings to SIRAP's private repository.

### 16.1 Generate the camera bridge for the camera PC

This cell writes a small Python bridge script into the temporary Colab workspace; it does not connect to a camera or start a tunnel. Download the generated file to the PC that can reach the camera, install Python and OpenCV there, and start the bridge from a terminal.

The bridge supports a USB webcam, an IP camera that OpenCV can read, or a local video file. It binds to 127.0.0.1 by default and exposes three token-protected routes: /snapshot.jpg for one frame, /video for an MJPEG stream, and /status for camera health. It stores its random bearer token in a local file under the PC user's home directory so it survives restarts. The token is not written into this notebook.

1. Run the script on the camera PC, for example with source 0 for its first webcam. For an IP camera, supply the RTSP URL through a local environment variable rather than placing a camera password in notebook text or shell history.
2. Read the token from the local token file and add it to the Colab Secret named CAMARA_TOKEN.
3. On the same PC, start [Cloudflare Tunnel](https://developers.cloudflare.com/tunnel/get-started/quick-tunnels/) with the local bridge address. A Quick Tunnel provides a temporary trycloudflare.com address without requiring an account or domain; use a named tunnel and Cloudflare Access for a stable, controlled deployment.
4. Add that base HTTPS address to the Colab Secret CAMARA_URL. Keep the address and token in Secrets; do not place either in notebook output or a public commit.

Install cloudflared by following [Cloudflare's official installation guide](https://developers.cloudflare.com/cloudflare-one/connections/connect-networks/downloads/), then run the command cloudflared tunnel --url http://127.0.0.1:8080. The URL changes when a Quick Tunnel is restarted. If the tunnel cannot connect, check that the camera PC can make outbound connections on port 7844 using QUIC/UDP or HTTP/2/TCP, as described in [Cloudflare's firewall guidance](https://developers.cloudflare.com/cloudflare-one/networks/connectors/cloudflare-tunnel/configure-tunnels/tunnel-with-firewall/).

For a tunnel protected by Cloudflare Access, also add CF_ACCESS_CLIENT_ID and CF_ACCESS_CLIENT_SECRET to Colab Secrets. The bridge token remains required as a second check.

**Recommendation.** Use a Quick Tunnel only for a temporary experiment. Use a named tunnel with Cloudflare Access for a stable remote test. For routine production operation, place inference on an on-site computer and transmit only the required recognition events over an authenticated API; this avoids continuous public video streaming and reduces latency and bandwidth.

In [ ]:
from pathlib import Path

BRIDGE_SCRIPT_PATH = Path("/content/SIRAP_workspace/live_camera/bridge_camera.py")
BRIDGE_SCRIPT_PATH.parent.mkdir(parents=True, exist_ok=True)

BRIDGE_SCRIPT = r'''
import argparse
import cv2
import hmac
import json
import os
import secrets
import threading
import time
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
from pathlib import Path

class FrameHub:
    def __init__(self, source, max_side, quality, fps, loop):
        self.source = source
        self.max_side = max_side
        self.quality = quality
        self.delay = 1.0 / max(fps, 1)
        self.loop = loop
        self.lock = threading.Lock()
        self.jpeg = None
        self.frame_id = 0
        self.captured_at = None
        self.width = 0
        self.height = 0
        self.last_error = ""
        self.stop_event = threading.Event()

    def start(self):
        threading.Thread(target=self._capture_loop, daemon=True).start()

    def snapshot(self):
        with self.lock:
            return self.jpeg, self.captured_at, self.frame_id

    def status(self):
        with self.lock:
            return {
                "ready": self.jpeg is not None,
                "width": self.width,
                "height": self.height,
                "frame_id": self.frame_id,
                "captured_at": self.captured_at,
                "last_error": self.last_error,
            }

    def _capture_loop(self):
        while not self.stop_event.is_set():
            capture = cv2.VideoCapture(self.source)
            capture.set(cv2.CAP_PROP_BUFFERSIZE, 1)
            if not capture.isOpened():
                with self.lock:
                    self.last_error = "OpenCV could not open the configured camera source."
                capture.release()
                self.stop_event.wait(2)
                continue

            source_is_file = isinstance(self.source, str) and Path(self.source).is_file()
            while not self.stop_event.is_set():
                started = time.monotonic()
                ok, frame = capture.read()
                if not ok:
                    if source_is_file and self.loop:
                        capture.set(cv2.CAP_PROP_POS_FRAMES, 0)
                        continue
                    if not source_is_file:
                        with self.lock:
                            self.last_error = "Camera frame read failed; reconnecting."
                        break
                    with self.lock:
                        self.last_error = "The video file reached its end."
                    self.stop_event.set()
                    break

                height, width = frame.shape[:2]
                scale = min(1.0, self.max_side / max(height, width))
                if scale < 1.0:
                    frame = cv2.resize(
                        frame,
                        (max(1, int(width * scale)), max(1, int(height * scale))),
                        interpolation=cv2.INTER_AREA,
                    )
                ok, encoded = cv2.imencode(
                    ".jpg", frame, [cv2.IMWRITE_JPEG_QUALITY, self.quality]
                )
                if ok:
                    with self.lock:
                        self.jpeg = encoded.tobytes()
                        self.frame_id += 1
                        self.captured_at = time.time()
                        self.height, self.width = frame.shape[:2]
                        self.last_error = ""
                remaining = self.delay - (time.monotonic() - started)
                if remaining > 0:
                    self.stop_event.wait(remaining)
            capture.release()
            if not source_is_file and not self.stop_event.is_set():
                self.stop_event.wait(1)

class Handler(BaseHTTPRequestHandler):
    hub = None
    token = ""

    def log_message(self, format, *args):
        return

    def _authorized(self):
        supplied = self.headers.get("X-Camera-Token", "")
        return hmac.compare_digest(supplied, self.token)

    def _send_json(self, value, status=200):
        body = json.dumps(value).encode("utf-8")
        self.send_response(status)
        self.send_header("Content-Type", "application/json")
        self.send_header("Content-Length", str(len(body)))
        self.send_header("Cache-Control", "no-store")
        self.end_headers()
        self.wfile.write(body)

    def do_GET(self):
        if not self._authorized():
            self._send_json({"error": "unauthorized"}, 401)
            return

        if self.path == "/status":
            self._send_json(self.hub.status())
            return

        if self.path == "/snapshot.jpg":
            jpeg, captured_at, frame_id = self.hub.snapshot()
            if jpeg is None:
                self._send_json({"error": "camera frame is not ready"}, 503)
                return
            self.send_response(200)
            self.send_header("Content-Type", "image/jpeg")
            self.send_header("Content-Length", str(len(jpeg)))
            self.send_header("X-Frame-Id", str(frame_id))
            self.send_header("X-Captured-At", str(captured_at))
            self.send_header("Cache-Control", "no-store")
            self.end_headers()
            self.wfile.write(jpeg)
            return

        if self.path == "/video":
            self.send_response(200)
            self.send_header("Content-Type", "multipart/x-mixed-replace; boundary=frame")
            self.send_header("Cache-Control", "no-store")
            self.send_header("Connection", "close")
            self.end_headers()
            try:
                last_id = -1
                while True:
                    jpeg, captured_at, frame_id = self.hub.snapshot()
                    if jpeg is None or frame_id == last_id:
                        time.sleep(0.02)
                        continue
                    last_id = frame_id
                    part = (
                        b"--frame\r\nContent-Type: image/jpeg\r\n"
                        + b"Content-Length: " + str(len(jpeg)).encode("ascii") + b"\r\n"
                        + b"X-Captured-At: " + str(captured_at).encode("ascii") + b"\r\n\r\n"
                        + jpeg + b"\r\n"
                    )
                    self.wfile.write(part)
                    self.wfile.flush()
            except (BrokenPipeError, ConnectionResetError, OSError):
                return
            return

        self._send_json({"error": "not found"}, 404)

def load_or_create_token(path):
    path = Path(path).expanduser()
    if path.exists():
        token = path.read_text(encoding="utf-8").strip()
        if token:
            return token, path
    path.parent.mkdir(parents=True, exist_ok=True)
    token = secrets.token_urlsafe(32)
    path.write_text(token + "\n", encoding="utf-8")
    try:
        path.chmod(0o600)
    except OSError:
        pass
    return token, path

def parse_source(value):
    return int(value) if value.isdigit() else value

def main():
    parser = argparse.ArgumentParser(description="Token-protected SIRAP camera bridge")
    parser.add_argument("--source", default=os.environ.get("SIRAP_CAMERA_SOURCE", "0"))
    parser.add_argument("--host", default="127.0.0.1")
    parser.add_argument("--port", type=int, default=8080)
    parser.add_argument("--max-side", type=int, default=1920)
    parser.add_argument("--quality", type=int, default=85)
    parser.add_argument("--fps", type=float, default=15)
    parser.add_argument("--loop", action="store_true", help="Loop a local video file.")
    args = parser.parse_args()

    token, token_path = load_or_create_token(Path.home() / ".sirap_camera_token")
    hub = FrameHub(
        parse_source(args.source), max(320, args.max_side),
        min(100, max(30, args.quality)), max(1, args.fps), args.loop,
    )
    hub.start()
    Handler.hub = hub
    Handler.token = token
    server = ThreadingHTTPServer((args.host, args.port), Handler)
    server.daemon_threads = True
    print("Camera bridge listening on http://127.0.0.1:" + str(args.port))
    print("Token file:", token_path)
    print("Routes: /snapshot.jpg, /video, /status")
    print("Stop with Ctrl+C. Keep this terminal open while the tunnel is running.")
    try:
        server.serve_forever()
    except KeyboardInterrupt:
        pass
    finally:
        hub.stop_event.set()
        server.server_close()

if __name__ == "__main__":
    main()
'''

BRIDGE_SCRIPT_PATH.write_text(BRIDGE_SCRIPT, encoding="utf-8")
print("Camera bridge script created:", BRIDGE_SCRIPT_PATH)
print("Download it from the Colab Files panel and copy it to the camera PC.")
print("On that PC, install OpenCV with: python -m pip install opencv-python")

### 16.2 Configure camera reading and the USA recognition pipeline

This cell defines the camera reader, the frame-processing settings, and the output folder. It does not connect to the camera or load the detector. The camera URL and token are read from Colab Secrets; optional Cloudflare Access service-token values are sent in request headers.

In automatic mode, an HTTP Cloudflare bridge is read as MJPEG first. If that stream does not deliver frames, the reader falls back to repeated snapshot requests. A separate background thread keeps only the latest frame, so slow CPU inference does not create a growing queue or increasing delay. Use OpenCV mode for a source that Colab can reach directly, such as a local video file, RTSP stream, or HLS playlist.

The processing function uses the loaded USA OpenVINO IR and the USA OCR settings. When a matching validation threshold record exists, it uses that threshold. Otherwise, it uses the documented provisional default of 0.25 and prints that the threshold was not calibrated. Live frames are not tracked across vehicles, and this notebook section does not make authorization or barrier decisions.

In [ ]:
import os
import json
import threading
import time
from urllib.parse import urlsplit, urlunsplit

import cv2
import numpy as np
import requests
import yaml
from IPython.display import display
from PIL import Image

def read_colab_secret(name):
    try:
        from google.colab import userdata
        value = userdata.get(name)
        if value:
            return str(value)
    except Exception:
        pass
    return os.environ.get(name, "")

LIVE_SOURCE = read_colab_secret("CAMARA_URL").strip()
LIVE_TOKEN = read_colab_secret("CAMARA_TOKEN").strip()
CF_ACCESS_CLIENT_ID = read_colab_secret("CF_ACCESS_CLIENT_ID").strip()
CF_ACCESS_CLIENT_SECRET = read_colab_secret("CF_ACCESS_CLIENT_SECRET").strip()

LIVE_READ_MODE = "auto"      # auto, mjpeg, snapshot, or opencv
LIVE_MJPEG_WAIT_S = 6
LIVE_SNAPSHOT_FPS = 4
LIVE_DURATION_S = 120        # None means run until the cell is interrupted.
LIVE_START_TIMEOUT_S = 30
LIVE_STALE_S = 20
LIVE_MAX_PROCESS_FPS = 5
LIVE_DISPLAY_FPS = 2
LIVE_CONFIRMATIONS = 2
LIVE_CONFIRM_WINDOW_S = 3
LIVE_DUPLICATE_COOLDOWN_S = 15
LIVE_SAVE_VIDEO = False
LIVE_OUTPUT_DIR = Path("/content/SIRAP_workspace/live_camera/usa")
LIVE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

class CameraUnavailable(RuntimeError):
    pass

def _camera_error(response):
    messages = {
        401: "The bridge rejected the camera token. Check the Colab Secret CAMARA_TOKEN.",
        403: "The request was denied by the bridge or Cloudflare Access. Check the configured Secrets.",
        404: "The bridge route was not found. Use the base tunnel URL in CAMARA_URL.",
        502: "Cloudflare could not reach the camera bridge. Confirm both processes are running.",
        530: "Cloudflare could not establish the requested connection. Check the tunnel and Access settings.",
    }
    return messages.get(
        response.status_code,
        "Camera request failed with HTTP " + str(response.status_code),
    )

class CameraReader:
    def __init__(self, source, token="", mode="auto", snapshot_fps=4,
                 access_client_id="", access_client_secret="", mjpeg_wait_s=6):
        self.source = source.strip()
        self.token = token
        self.mode = mode.lower()
        self.snapshot_fps = max(1.0, float(snapshot_fps))
        self.mjpeg_wait_s = max(1.0, float(mjpeg_wait_s))
        self.access_client_id = access_client_id
        self.access_client_secret = access_client_secret
        self.stop_event = threading.Event()
        self.lock = threading.Lock()
        self.frame = None
        self.received = 0
        self.last_frame_monotonic = None
        self.last_frame_epoch = None
        self.error = ""
        self.thread = None
        self.base_url = self._base_url(self.source)

    @staticmethod
    def _base_url(source):
        parts = urlsplit(source)
        if parts.scheme not in ("http", "https"):
            return source
        path = parts.path.rstrip("/")
        for suffix in ("/video", "/snapshot.jpg", "/status"):
            if path.endswith(suffix):
                path = path[:-len(suffix)]
                break
        return urlunsplit((parts.scheme, parts.netloc, path, "", "")).rstrip("/")

    def _headers(self):
        headers = {}
        if self.token:
            headers["X-Camera-Token"] = self.token
        if self.access_client_id:
            headers["CF-Access-Client-Id"] = self.access_client_id
        if self.access_client_secret:
            headers["CF-Access-Client-Secret"] = self.access_client_secret
        return headers

    def start(self):
        if not self.source:
            raise CameraUnavailable("No camera URL is configured. Add CAMARA_URL to Colab Secrets.")
        self.thread = threading.Thread(target=self._run, daemon=True)
        self.thread.start()
        return self

    def stop(self):
        self.stop_event.set()
        if self.thread:
            self.thread.join(timeout=3)

    def snapshot(self):
        with self.lock:
            if self.frame is None:
                return None, self.received, self.last_frame_monotonic, self.last_frame_epoch
            return self.frame.copy(), self.received, self.last_frame_monotonic, self.last_frame_epoch

    def _offer(self, frame):
        if frame is None or frame.size == 0:
            return
        with self.lock:
            self.frame = frame
            self.received += 1
            self.last_frame_monotonic = time.monotonic()
            self.last_frame_epoch = time.time()
            self.error = ""

    def _read_opencv(self):
        source = int(self.source) if self.source.isdigit() else self.source
        capture = cv2.VideoCapture(source)
        capture.set(cv2.CAP_PROP_BUFFERSIZE, 1)
        if not capture.isOpened():
            capture.release()
            raise CameraUnavailable("OpenCV could not open the configured source.")
        try:
            while not self.stop_event.is_set():
                ok, frame = capture.read()
                if not ok:
                    raise CameraUnavailable("OpenCV stopped receiving frames from the source.")
                self._offer(frame)
        finally:
            capture.release()

    def _read_mjpeg(self):
        url = self.base_url + "/video"
        response = requests.get(
            url, headers=self._headers(), stream=True, timeout=(8, 4)
        )
        if response.status_code >= 400:
            message = _camera_error(response)
            response.close()
            raise CameraUnavailable(message)
        before = self.received
        started = time.monotonic()
        buffer = bytearray()
        jpeg_start = bytes([255, 216])
        jpeg_end = bytes([255, 217])
        try:
            for chunk in response.iter_content(chunk_size=2048):
                if self.stop_event.is_set():
                    break
                if not chunk:
                    continue
                buffer.extend(chunk)
                while True:
                    start = buffer.find(jpeg_start)
                    end = buffer.find(jpeg_end, start + 2) if start >= 0 else -1
                    if start < 0 or end < 0:
                        if len(buffer) > 5_000_000:
                            del buffer[:-2]
                        break
                    jpeg = bytes(buffer[start:end + 2])
                    del buffer[:end + 2]
                    frame = cv2.imdecode(np.frombuffer(jpeg, dtype=np.uint8), cv2.IMREAD_COLOR)
                    self._offer(frame)
                if self.received == before and time.monotonic() - started >= self.mjpeg_wait_s:
                    return False
        finally:
            response.close()
        return self.received > before

    def _read_snapshots(self):
        url = self.base_url + "/snapshot.jpg"
        delay = 1.0 / self.snapshot_fps
        while not self.stop_event.is_set():
            started = time.monotonic()
            response = requests.get(url, headers=self._headers(), timeout=(8, 10))
            if response.status_code >= 400:
                message = _camera_error(response)
                response.close()
                raise CameraUnavailable(message)
            frame = cv2.imdecode(
                np.frombuffer(response.content, dtype=np.uint8), cv2.IMREAD_COLOR
            )
            response.close()
            if frame is None:
                raise CameraUnavailable("The snapshot endpoint did not return a valid JPEG.")
            self._offer(frame)
            self.stop_event.wait(max(0.0, delay - (time.monotonic() - started)))

    def _run(self):
        if self.mode == "opencv" or urlsplit(self.source).scheme not in ("http", "https"):
            try:
                self._read_opencv()
            except Exception as exc:
                self.error = str(exc)
            return

        while not self.stop_event.is_set():
            try:
                if self.mode == "snapshot":
                    self._read_snapshots()
                    return
                try:
                    received_mjpeg = self._read_mjpeg()
                except Exception as exc:
                    self.error = str(exc)
                    if self.mode != "auto":
                        return
                    received_mjpeg = False
                if self.mode == "mjpeg":
                    if not received_mjpeg:
                        self.error = "No MJPEG frames arrived before the timeout."
                        return
                    continue
                if not received_mjpeg:
                    self._read_snapshots()
                    return
            except Exception as exc:
                self.error = str(exc)
                if self.mode != "auto":
                    return
                self.stop_event.wait(2)

def open_camera():
    return CameraReader(
        LIVE_SOURCE, LIVE_TOKEN, LIVE_READ_MODE, LIVE_SNAPSHOT_FPS,
        CF_ACCESS_CLIENT_ID, CF_ACCESS_CLIENT_SECRET, LIVE_MJPEG_WAIT_S,
    )

def build_live_pipeline():
    if "DETECTOR_DIR" not in globals() or "MODEL_ID" not in globals():
        raise RuntimeError("Run section 13 first to select the USA OpenVINO IR.")
    if not Path(DETECTOR_DIR).exists():
        raise FileNotFoundError("The selected OpenVINO IR directory is missing.")

    from src.data_processing import DEFAULT_PATTERNS
    from src.model import ALPRPipeline, PlateDetector, PlateReader

    threshold_path = METRICS_DIR / "rtdetr_usa_threshold_calibration.json"
    live_confidence = 0.25
    threshold_source = "provisional default (0.25)"
    if threshold_path.is_file():
        record = json.loads(threshold_path.read_text(encoding="utf-8"))
        if (
            record.get("checkpoint") != str(COUNTRY_IR_DIR)
            or record.get("model_selection_record") != MODEL_SELECTION_RECORD
        ):
            raise RuntimeError("The saved threshold does not match the selected USA checkpoint.")
        live_confidence = float(record["threshold"])
        threshold_source = "validation calibration"

    usa_config = yaml.safe_load((REPO_ROOT / "config/countries/usa.yaml").read_text(encoding="utf-8"))
    detector_config = usa_config["detector"]
    ocr_config = usa_config["ocr"]
    live_detector = PlateDetector(
        checkpoint=DETECTOR_DIR,
        model_id=MODEL_ID,
        imgsz=IMAGE_SIZE,
        conf=live_confidence,
        class_id=detector_config.get("class_id", 0),
        device="CPU",
        fallback_checkpoint=None,
    )
    reader = PlateReader(
        model_name=ocr_config.get("modelo", "cct-s-v2-global-model"),
        device=ocr_config.get("dispositivo", "cpu"),
        onnx_path=ocr_config.get("onnx_afinado"),
        plate_config_path=ocr_config.get("plate_config_afinado"),
    )
    patterns = [
        tuple(pattern) for pattern in usa_config.get("placas", {}).get("patrones", DEFAULT_PATTERNS)
    ]
    pipeline = ALPRPipeline(
        live_detector, reader, patterns=patterns,
        margin=detector_config.get("margen_recorte", 0.06),
    )
    return pipeline, live_confidence, threshold_source

def annotate_live_frame(frame_bgr, recognition, camera_fps, process_fps):
    annotated = frame_bgr.copy()
    if recognition.box:
        x1, y1, x2, y2 = (int(round(value)) for value in recognition.box)
        height, width = annotated.shape[:2]
        x1, x2 = max(0, min(width - 1, x1)), max(0, min(width - 1, x2))
        y1, y2 = max(0, min(height - 1, y1)), max(0, min(height - 1, y2))
        cv2.rectangle(annotated, (x1, y1), (x2, y2), (0, 220, 0), 2)
        label = recognition.plate or recognition.plate_raw or "Plate candidate"
        cv2.putText(
            annotated, str(label)[:32], (x1, max(24, y1 - 8)),
            cv2.FONT_HERSHEY_SIMPLEX, 0.65, (0, 220, 0), 2, cv2.LINE_AA,
        )
    cv2.putText(
        annotated, f"Camera {camera_fps:.1f} fps | Processing {process_fps:.1f} fps",
        (12, 26), cv2.FONT_HERSHEY_SIMPLEX, 0.58, (255, 255, 255), 2, cv2.LINE_AA,
    )
    return annotated

print("Live-camera configuration defined; no connection has been opened.")
print("Camera URL configured:", bool(LIVE_SOURCE))
print("Reader mode:", LIVE_READ_MODE)
print("Live outputs:", LIVE_OUTPUT_DIR)
print("The selected model is USA fine-tuned:", globals().get("MODEL_IS_COUNTRY_FINETUNED", False))
if not globals().get("MODEL_IS_COUNTRY_FINETUNED", False):
    print("Warning: the shared checkpoint is a technical fallback, not evidence of USA accuracy.")

### 16.3 Test the camera connection

This check connects for five seconds without loading the detector or OCR model. It reports the number of frames received, approximate frame rate, resolution, and any connection error, then displays the last frame so the camera angle can be checked. It does not save a video or score recognition accuracy.

In [ ]:
LIVE_TEST_SECONDS = 5

if not LIVE_SOURCE:
    print("No camera URL is configured. Add CAMARA_URL and CAMARA_TOKEN to Colab Secrets, then rerun section 16.2.")
else:
    test_camera = open_camera()
    try:
        test_camera.start()
        start_time = time.monotonic()
        first_count = test_camera.received
        latest = None
        while time.monotonic() - start_time < LIVE_TEST_SECONDS:
            frame, count, _, _ = test_camera.snapshot()
            if frame is not None:
                latest = frame
            time.sleep(0.05)
        elapsed = max(time.monotonic() - start_time, 1e-6)
        final_count = test_camera.received
        if latest is None:
            detail = test_camera.error or "No frames arrived during the connection check."
            raise CameraUnavailable(detail)
        print({
            "frames_received": final_count - first_count,
            "approx_camera_fps": round((final_count - first_count) / elapsed, 2),
            "resolution": f"{latest.shape[1]} x {latest.shape[0]}",
            "reader_mode": LIVE_READ_MODE,
            "elapsed_seconds": round(elapsed, 2),
        })
        display(Image.fromarray(cv2.cvtColor(latest, cv2.COLOR_BGR2RGB)))
    except CameraUnavailable as exc:
        print("Camera connection check failed:", exc)
    finally:
        test_camera.stop()

### 16.4 Run the live session and save its outputs

Set LIVE_ENABLED to True only when you deliberately want to connect to the camera. The loop processes the newest available frame, applies the USA detector and OCR, and updates the preview. A reading is added to the session CSV after the same text is seen at least twice within the confirmation window; repeated readings are suppressed for the configured cooldown.

The session stops at LIVE_DURATION_S, when the camera has been stale for too long, or when you interrupt the cell. The output folder contains a CSV of confirmed text readings and a JSON session summary. Set LIVE_SAVE_VIDEO to True to also write an annotated MP4. The live loop does not use object tracking, an authorization list, or a gate action. It reports recognition evidence for review; it does not produce access decisions.

The CSV, summary, and optional video may contain personal data. They are saved outside the cloned repository in Colab's temporary workspace. Review them only for an authorized purpose and remove them when no longer needed.

In [ ]:
from collections import defaultdict, deque
from datetime import datetime, timezone
import csv
import json

LIVE_ENABLED = False

def run_live_session():
    pipeline, live_confidence, threshold_source = build_live_pipeline()
    reader = open_camera()
    session_id = datetime.now(timezone.utc).strftime("usa_live_%Y%m%dT%H%M%S%fZ")
    session_dir = LIVE_OUTPUT_DIR / session_id
    session_dir.mkdir(parents=True, exist_ok=False)
    readings_path = session_dir / "readings.csv"
    summary_path = session_dir / "summary.json"
    video_path = session_dir / "annotated.mp4"

    observations = defaultdict(lambda: deque(maxlen=LIVE_CONFIRMATIONS))
    last_logged = {}
    rows = []
    latencies_ms = []
    writer = None
    display_handle = None
    started = time.monotonic()
    last_processed_id = -1
    last_processed_at = 0.0
    last_display_at = 0.0
    processed = 0

    if LIVE_SAVE_VIDEO:
        print("Annotated video recording is enabled.")

    reader.start()
    try:
        initial_deadline = time.monotonic() + LIVE_START_TIMEOUT_S
        while reader.received == 0 and time.monotonic() < initial_deadline:
            if reader.error and LIVE_READ_MODE != "auto":
                raise CameraUnavailable(reader.error)
            time.sleep(0.1)
        if reader.received == 0:
            raise CameraUnavailable(reader.error or "No camera frame arrived before the startup timeout.")

        while True:
            now = time.monotonic()
            if LIVE_DURATION_S is not None and now - started >= LIVE_DURATION_S:
                break
            frame_bgr, frame_id, last_frame_at, captured_epoch = reader.snapshot()
            if frame_bgr is None:
                time.sleep(0.02)
                continue
            if last_frame_at is not None and now - last_frame_at > LIVE_STALE_S:
                raise CameraUnavailable("The camera stream stopped sending frames.")
            if frame_id == last_processed_id:
                time.sleep(0.01)
                continue
            if now - last_processed_at < 1.0 / max(LIVE_MAX_PROCESS_FPS, 0.1):
                time.sleep(0.01)
                continue

            last_processed_id = frame_id
            last_processed_at = now
            frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
            inference_started = time.perf_counter()
            recognition = pipeline.recognize(frame_rgb)
            inference_ms = (time.perf_counter() - inference_started) * 1000
            latencies_ms.append(inference_ms)
            processed += 1

            elapsed_now = max(now - started, 1e-6)
            camera_fps = reader.received / elapsed_now
            process_fps = processed / elapsed_now
            annotated = annotate_live_frame(frame_bgr, recognition, camera_fps, process_fps)

            raw_text = (recognition.plate or recognition.plate_raw or "").strip().upper()
            normalized_text = (recognition.plate or "").strip()
            if raw_text:
                history = observations[raw_text]
                while history and now - history[0] > LIVE_CONFIRM_WINDOW_S:
                    history.popleft()
                history.append(now)
                last_time = last_logged.get(raw_text, 0.0)
                if len(history) >= LIVE_CONFIRMATIONS and now - last_time >= LIVE_DUPLICATE_COOLDOWN_S:
                    rows.append({
                        "timestamp_utc": datetime.fromtimestamp(
                            captured_epoch or time.time(), tz=timezone.utc
                        ).isoformat(),
                        "plate": normalized_text,
                        "plate_raw": recognition.plate_raw,
                        "detector_confidence": recognition.det_confidence,
                        "ocr_confidence_min": recognition.ocr_confidence,
                        "ocr_confidence_mean": recognition.ocr_mean_confidence,
                        "box_xyxy": json.dumps(recognition.box),
                        "confirmation_count": len(history),
                        "model": "RT-DETRv2-S",
                        "runtime": "OpenVINO Runtime",
                    })
                    last_logged[raw_text] = now

            if LIVE_SAVE_VIDEO:
                height, width = annotated.shape[:2]
                if writer is None:
                    writer = cv2.VideoWriter(
                        str(video_path), cv2.VideoWriter_fourcc(*"mp4v"),
                        max(1.0, min(float(LIVE_MAX_PROCESS_FPS), 10.0)), (width, height),
                    )
                    if not writer.isOpened():
                        writer.release()
                        writer = None
                        raise RuntimeError("Could not create the annotated MP4 output.")
                writer.write(annotated)

            if now - last_display_at >= 1.0 / max(LIVE_DISPLAY_FPS, 0.1):
                preview = Image.fromarray(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
                if display_handle is None:
                    display_handle = display(preview, display_id=True)
                else:
                    display_handle.update(preview)
                last_display_at = now
    except KeyboardInterrupt:
        print("Live session interrupted by the user; saving the readings collected so far.")
    finally:
        reader.stop()
        if writer is not None:
            writer.release()

    elapsed = max(time.monotonic() - started, 1e-6)
    readings_df = pd.DataFrame(
        rows,
        columns=[
            "timestamp_utc", "plate", "plate_raw", "detector_confidence",
            "ocr_confidence_min", "ocr_confidence_mean", "box_xyxy",
            "confirmation_count", "model", "runtime",
        ],
    )
    readings_df.to_csv(readings_path, index=False)
    summary = {
        "session_id": session_id,
        "country": "USA",
        "model": "RT-DETRv2-S",
        "runtime": "OpenVINO Runtime",
        "country_finetuned_checkpoint": bool(globals().get("MODEL_IS_COUNTRY_FINETUNED", False)),
        "confidence_threshold": live_confidence,
        "threshold_source": threshold_source,
        "duration_seconds": round(elapsed, 2),
        "frames_received": int(reader.received),
        "frames_processed": int(processed),
        "camera_fps": round(reader.received / elapsed, 3),
        "processing_fps": round(processed / elapsed, 3),
        "mean_pipeline_latency_ms": (
            round(float(np.mean(latencies_ms)), 2) if latencies_ms else None
        ),
        "confirmed_readings": len(rows),
        "readings_csv": str(readings_path),
        "annotated_video": str(video_path) if LIVE_SAVE_VIDEO and video_path.is_file() else None,
        "authorization_decisions": False,
    }
    summary_path.write_text(json.dumps(summary, indent=2), encoding="utf-8")
    print("Live session summary:")
    display(pd.Series(summary, name=session_id).to_frame())
    print("Confirmed readings:")
    display(readings_df)
    print("Session files:", session_dir)
    if not globals().get("MODEL_IS_COUNTRY_FINETUNED", False):
        print("Reminder: this run used the shared technical fallback, not a USA-fine-tuned checkpoint.")
    return summary

live_summary = None
if not LIVE_ENABLED:
    print("Live processing is disabled. Set LIVE_ENABLED = True to run a deliberate camera session.")
elif not LIVE_SOURCE:
    print("Live processing skipped: add CAMARA_URL and CAMARA_TOKEN to Colab Secrets first.")
else:
    try:
        live_summary = run_live_session()
    except CameraUnavailable as exc:
        print("Live camera session stopped:", exc)


## 17. Run reproducible image robustness checks

This diagnostic applies synthetic low-light, high-light, blur, half-resolution, and JPEG-compression changes to up to 200 validation images. It compares precision, recall, F1, and counts at the validation-selected threshold. The detector test split is not used here, so it remains available for the final one-time evaluation. Synthetic stress conditions do not replace real USA scenes across weather, lighting, distance, camera angles, and plate designs.

In [ ]:
import cv2

def darker(image):
    return np.clip(image.astype(np.float32) * 0.55, 0, 255).astype(np.uint8)

def brighter(image):
    return np.clip(image.astype(np.float32) * 1.25 + 12, 0, 255).astype(np.uint8)

def blurred(image):
    return cv2.GaussianBlur(image, (5, 5), 0)

def reduced(image):
    h, w = image.shape[:2]
    small = cv2.resize(image, (max(1, w // 2), max(1, h // 2)))
    return cv2.resize(small, (w, h), interpolation=cv2.INTER_LINEAR)

def compressed(image):
    bgr = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)
    ok, encoded = cv2.imencode(".jpg", bgr, [cv2.IMWRITE_JPEG_QUALITY, 35])
    if not ok:
        return image
    decoded = cv2.imdecode(encoded, cv2.IMREAD_COLOR)
    return cv2.cvtColor(decoded, cv2.COLOR_BGR2RGB)

RUN_ROBUSTNESS_TEST = False
ACTIVE_THRESHOLD = 0.25
if RUN_ROBUSTNESS_TEST:
    if not DATA_READY or not COUNTRY_IR_XML.is_file():
        raise RuntimeError("A COCO validation split and the USA fine-tuned IR are required.")
    threshold_file = METRICS_DIR / "rtdetr_usa_threshold_calibration.json"
    if not threshold_file.is_file():
        raise RuntimeError("Calibrate the USA confidence threshold on validation first.")
    threshold_record = json.loads(threshold_file.read_text(encoding="utf-8"))
    if (
        threshold_record.get("checkpoint") != str(COUNTRY_IR_DIR)
        or threshold_record.get("model_selection_record") != MODEL_SELECTION_RECORD
    ):
        raise RuntimeError("The saved threshold does not match the selected USA checkpoint.")
    ACTIVE_THRESHOLD = float(threshold_record["threshold"])
    variants = {
        "original": None,
        "low_light": darker,
        "high_light": brighter,
        "blur": blurred,
        "half_resolution": reduced,
        "jpeg_compression": compressed,
    }
    stress_rows = []
    for name, transform in variants.items():
        records = collect_detection_records("val", max_images=200, transform=transform)
        stress_rows.append({"condition": name, **score_detection_records(records, ACTIVE_THRESHOLD)})
    robustness_table = pd.DataFrame(stress_rows)
    display(robustness_table)
    robustness_table.to_csv(METRICS_DIR / "image_robustness_rtdetr_usa.csv", index=False)
else:
    print("Stress testing is disabled. It does not change the model or production threshold.")

## 18. Measure CPU inference latency

This timing check measures detector-only inference on up to 200 validation images after warming up with the first ten. It reports median, p95, p99, and approximate frames per second from the median. Image reading, OCR, network transfer, and application logic are excluded. Use the same CPU class and input size when comparing these figures with the deployed Space.

In [ ]:
RUN_LATENCY_TEST = False
if RUN_LATENCY_TEST:
    if not DATA_READY or not COUNTRY_IR_XML.is_file():
        raise RuntimeError("A COCO validation split and the USA fine-tuned IR are required.")
    validation_dataset = SPLITS["val"]["dataset"]
    samples = validation_dataset.images[:min(200, len(validation_dataset.images))]
    for item in samples[:10]:
        image = load_image(validation_dataset._image_path(item["file_name"]).read_bytes())
        detector.detect(image, max_detections=100)
    latencies_ms = []
    for item in samples[10:]:
        image = load_image(validation_dataset._image_path(item["file_name"]).read_bytes())
        start = time.perf_counter()
        detector.detect(image, max_detections=100)
        latencies_ms.append((time.perf_counter() - start) * 1000)
    latency_summary = {
        "n": len(latencies_ms),
        "device": "CPU",
        "median_ms": float(np.median(latencies_ms)) if latencies_ms else None,
        "p95_ms": float(np.percentile(latencies_ms, 95)) if latencies_ms else None,
        "p99_ms": float(np.percentile(latencies_ms, 99)) if latencies_ms else None,
        "fps_from_median": 1000 / float(np.median(latencies_ms)) if latencies_ms else None,
    }
    (METRICS_DIR / "openvino_usa_latency.json").write_text(
        json.dumps(latency_summary, indent=2), encoding="utf-8"
    )
    print(latency_summary)
else:
    print("Latency measurement is disabled.")

## 19. Evaluate OCR on labeled USA plate crops

This independent OCR test uses datasets/usa/seg_and_ocr. It reads groundtruth.csv and matching files in usimages, runs the global fast-plate-ocr recognizer on CPU, and reports exact-match rate and character error rate, including results by state. It evaluates OCR on already-cropped plates, so it does not measure detector misses. Run it only if the files are present.

In [ ]:
RUN_OCR_CROP_TEST = False
OCR_CSV = REPO_ROOT / "datasets/usa/seg_and_ocr/groundtruth.csv"
OCR_IMAGE_DIR = REPO_ROOT / "datasets/usa/seg_and_ocr/usimages"

if RUN_OCR_CROP_TEST:
    from fast_plate_ocr import LicensePlateRecognizer
    from src.data_processing import clean_text, plate_distance

    if not OCR_CSV.is_file() or not OCR_IMAGE_DIR.is_dir():
        raise FileNotFoundError("Could not find the USA profile CSV or OCR crop images.")
    ocr_truth = pd.read_csv(
        OCR_CSV, header=None, names=["image_file", "state", "plate_gt"], dtype=str
    ).dropna(subset=["image_file", "plate_gt"])
    recognizer = LicensePlateRecognizer("cct-s-v2-global-model", device="cpu")
    ocr_rows = []
    for row in ocr_truth.itertuples(index=False):
        image_path = OCR_IMAGE_DIR / row.image_file
        if not image_path.is_file():
            continue
        crop = np.asarray(Image.open(image_path).convert("RGB"))
        prediction = recognizer.run(crop, return_confidence=True)[0]
        pred = clean_text(prediction.plate)
        gt = clean_text(row.plate_gt)
        probs = np.asarray(prediction.char_probs, dtype=np.float32).ravel()
        ocr_rows.append({
            "image_file": row.image_file,
            "state": row.state,
            "plate_gt": gt,
            "plate_pred": pred,
            "exact_match": pred == gt,
            "char_errors": plate_distance(pred, gt),
            "gt_characters": len(gt),
            "confidence_min": float(probs.min()) if probs.size else 0.0,
        })
    ocr_results = pd.DataFrame(ocr_rows)
    emr = float(ocr_results["exact_match"].mean()) if len(ocr_results) else 0.0
    cer = (
        float(ocr_results["char_errors"].sum() / max(ocr_results["gt_characters"].sum(), 1))
        if len(ocr_results) else 0.0
    )
    print({"n": len(ocr_results), "exact_match_rate": emr, "character_error_rate": cer})
    display(ocr_results.groupby("state")["exact_match"].agg(["count", "mean"]).sort_values("mean"))
    ocr_results.to_csv(METRICS_DIR / "rtdetr_usa_crop_ocr.csv", index=False)
else:
    print("OCR crop evaluation is disabled.")

## 20. Optional evaluation with SIRAP's PlateReader

This additional evaluation uses SIRAP's PlateReader on a user-supplied CSV with image_path and plate_gt. It preserves raw reference text, raw prediction, minimum character confidence, and mean confidence for manual inspection. It does not calculate a USA accuracy score because country-specific normalization rules must be validated first. Keep it separate from detector mAP.

In [ ]:
import pandas as pd
from pathlib import Path
from src.data_processing import load_image
from src.model import PlateReader
CSV_PATH=None  # CSV with image_path and plate_gt; avoid unvalidated country-specific normalization
if CSV_PATH:
    reader=PlateReader(); data=pd.read_csv(CSV_PATH); rows=[]
    for row in data.itertuples(index=False):
        raw,conf,mean=reader.read(load_image(Path(row.image_path).read_bytes()))
        rows.append({"plate_gt_raw":str(row.plate_gt),"plate_pred_raw":raw,
                     "ocr_confidence":conf,"ocr_mean_confidence":mean})
    display(pd.DataFrame(rows).head())
else: print("Set CSV_PATH to inspect OCR crops.")


## 21. Optional end-to-end evaluation on full scenes

This separate evaluation measures detection plus OCR on complete USA scenes. The labeled CSV must contain image_path and plate_gt. Use a separate scene set that does not overlap training or the detector test. The cell filters detections with the validation-calibrated threshold, crops each candidate, runs OCR, and reports exact match, character error rate, and missed-detection rate. It does not apply access-control rules. The output guard prevents accidental overwriting.

In [ ]:
RUN_SCENE_E2E_TEST = False
SCENE_E2E_CSV = REPO_ROOT / "datasets/usa/scene_e2e_test.csv"
SCENE_E2E_OUTPUT_PATH = METRICS_DIR / "rtdetr_usa_scene_e2e.csv"

if RUN_SCENE_E2E_TEST:
    from fast_plate_ocr import LicensePlateRecognizer
    from src.data_processing import clean_text, crop_with_margin, plate_distance

    if not COUNTRY_IR_XML.is_file():
        raise RuntimeError("The USA fine-tuned IR is missing.")
    if not MODEL_CHECKPOINT_FROZEN:
        raise RuntimeError("Freeze the selected checkpoint before running the scene-level evaluation.")
    if SCENE_E2E_OUTPUT_PATH.exists():
        raise FileExistsError(f"Scene-level output already exists: {SCENE_E2E_OUTPUT_PATH}")
    threshold_file = METRICS_DIR / "rtdetr_usa_threshold_calibration.json"
    if not threshold_file.is_file():
        raise RuntimeError("Calibrate the USA confidence threshold on validation first.")
    threshold_record = json.loads(threshold_file.read_text(encoding="utf-8"))
    if (
        threshold_record.get("checkpoint") != str(COUNTRY_IR_DIR)
        or threshold_record.get("model_selection_record") != MODEL_SELECTION_RECORD
    ):
        raise RuntimeError("The saved threshold does not match the selected USA checkpoint.")
    ACTIVE_THRESHOLD = float(threshold_record["threshold"])
    if not SCENE_E2E_CSV.is_file():
        raise FileNotFoundError("Create scene_e2e_test.csv with image_path and plate_gt columns.")
    scene_truth = pd.read_csv(SCENE_E2E_CSV, dtype=str)
    if not {"image_path", "plate_gt"}.issubset(scene_truth.columns):
        raise ValueError("The CSV must contain image_path and plate_gt columns.")
    scene_truth = scene_truth.dropna(subset=["image_path", "plate_gt"])
    recognizer = LicensePlateRecognizer("cct-s-v2-global-model", device="cpu")
    predictions = []
    for row in scene_truth.itertuples(index=False):
        candidate_path = Path(row.image_path)
        if not candidate_path.is_absolute():
            options = [REPO_ROOT / candidate_path, Path(DATA_DIR) / candidate_path]
            candidate_path = next((p for p in options if p.is_file()), options[0])
        image = load_image(candidate_path.read_bytes())
        detections = [
            d for d in detector.detect(image, max_detections=20)
            if d.confidence >= ACTIVE_THRESHOLD
        ]
        candidates = []
        for detection in detections:
            crop = crop_with_margin(image, detection.box, 0.06)
            if crop.size == 0:
                continue
            read = recognizer.run(crop, return_confidence=True)[0]
            text = clean_text(read.plate)
            probs = np.asarray(read.char_probs, dtype=np.float32).ravel()
            confidence = float(probs.mean()) if probs.size else 0.0
            candidates.append((text, confidence, detection.confidence))
        best = max(candidates, key=lambda item: (item[1], item[2])) if candidates else ("", 0.0, 0.0)
        gt = clean_text(row.plate_gt)
        predictions.append({
            "image_path": str(candidate_path),
            "plate_gt": gt,
            "plate_pred": best[0],
            "exact_match": best[0] == gt,
            "char_errors": plate_distance(best[0], gt),
            "detected": bool(detections),
            "det_confidence": best[2],
            "ocr_confidence": best[1],
        })
    e2e_results = pd.DataFrame(predictions)
    print({
        "n": len(e2e_results),
        "exact_match_rate": float(e2e_results["exact_match"].mean()) if len(e2e_results) else 0.0,
        "character_error_rate": float(
            e2e_results["char_errors"].sum() / max(e2e_results["plate_gt"].str.len().sum(), 1)
        ) if len(e2e_results) else 0.0,
        "missed_detection_rate": float((~e2e_results["detected"]).mean()) if len(e2e_results) else 0.0,
    })
    e2e_results.to_csv(METRICS_DIR / "rtdetr_usa_scene_e2e.csv", index=False)
    display(e2e_results.head())
else:
    print("End-to-end test is disabled.")

## 22. Optional end-to-end evaluation with the SIRAP pipeline

This evaluation runs the actual SIRAP ALPRPipeline on a separately labeled set of complete scenes. It measures the detector and the configured SIRAP OCR reader together, unlike the crop-only OCR section. The CSV must contain image_path and plate_gt. The cell applies the validation-calibrated confidence threshold, writes English result columns and metrics, and refuses to overwrite an existing report. Use a scene set that is disjoint from training and the detector holdout test.

In [ ]:
RUN_SIRAP_PIPELINE_E2E = False
SIRAP_PIPELINE_E2E_CSV = REPO_ROOT / "datasets/usa/scene_e2e_test.csv"
SIRAP_PIPELINE_E2E_CSV_OUT = METRICS_DIR / "rtdetr_usa_sirap_pipeline_e2e.csv"
SIRAP_PIPELINE_E2E_JSON_OUT = METRICS_DIR / "rtdetr_usa_sirap_pipeline_e2e.json"

if RUN_SIRAP_PIPELINE_E2E:
    if not MODEL_CHECKPOINT_FROZEN or not COUNTRY_IR_XML.is_file():
        raise RuntimeError("Freeze the selected USA checkpoint before running pipeline evaluation.")
    if not SIRAP_PIPELINE_E2E_CSV.is_file():
        raise FileNotFoundError("The labeled full-scene CSV is missing.")
    threshold_file = METRICS_DIR / "rtdetr_usa_threshold_calibration.json"
    if not threshold_file.is_file():
        raise RuntimeError("Calibrate the USA confidence threshold on validation first.")
    threshold_record = json.loads(threshold_file.read_text(encoding="utf-8"))
    if (
        threshold_record.get("checkpoint") != str(COUNTRY_IR_DIR)
        or threshold_record.get("model_selection_record") != MODEL_SELECTION_RECORD
    ):
        raise RuntimeError("The saved threshold does not match the selected USA checkpoint.")
    ACTIVE_THRESHOLD = float(threshold_record["threshold"])
    if SIRAP_PIPELINE_E2E_CSV_OUT.exists() or SIRAP_PIPELINE_E2E_JSON_OUT.exists():
        raise FileExistsError("A SIRAP pipeline end-to-end report already exists.")
    scene_data = pd.read_csv(SIRAP_PIPELINE_E2E_CSV, dtype=str)
    if not {"image_path", "plate_gt"}.issubset(scene_data.columns):
        raise ValueError("The CSV must contain image_path and plate_gt columns.")

    from src.model import ALPRPipeline
    from src.data_processing import clean_text, load_image, plate_distance

    pipeline = ALPRPipeline.from_config()
    pipeline.detector.conf = ACTIVE_THRESHOLD
    rows = []
    for row in scene_data.dropna(subset=["image_path", "plate_gt"]).itertuples(index=False):
        image_path = Path(row.image_path)
        if not image_path.is_absolute():
            image_path = REPO_ROOT / image_path
        recognition = pipeline.recognize(load_image(image_path.read_bytes()))
        reference = clean_text(row.plate_gt)
        prediction = clean_text(recognition.plate or "")
        rows.append({
            "image_path": str(image_path),
            "plate_reference": reference,
            "plate_prediction": prediction,
            "exact_match": prediction == reference,
            "character_errors": plate_distance(prediction, reference),
            "detector_confidence": recognition.det_confidence,
            "detected": recognition.box is not None,
            "ocr_confidence": recognition.ocr_confidence,
            "latency_ms": recognition.timings_ms.get("total"),
        })

    pipeline_results = pd.DataFrame(rows)
    result_summary = {
        "detector": pipeline.detector.name,
        "checkpoint": pipeline.detector.source,
        "image_count": len(pipeline_results),
        "exact_match_rate": float(pipeline_results["exact_match"].mean()) if len(pipeline_results) else 0.0,
        "character_error_rate": float(
            pipeline_results["character_errors"].sum()
            / max(pipeline_results["plate_reference"].str.len().sum(), 1)
        ) if len(pipeline_results) else 0.0,
        "missed_detection_rate": float((~pipeline_results["detected"]).mean())
        if len(pipeline_results) else 0.0,
        "operational_confidence_threshold": ACTIVE_THRESHOLD,
    }
    pipeline_results.to_csv(SIRAP_PIPELINE_E2E_CSV_OUT, index=False)
    SIRAP_PIPELINE_E2E_JSON_OUT.write_text(json.dumps(result_summary, indent=2), encoding="utf-8")
    print(result_summary)
    display(pipeline_results.head())
else:
    print("SIRAP pipeline end-to-end evaluation is disabled.")

## 23. Evaluate the held-out detector test exactly once

This is the final COCO detector evaluation. It requires the grouped split audit, a written model-selection record, the country-specific OpenVINO IR, and at least 400 held-out test images. Before running it, set MODEL_CHECKPOINT_FROZEN to True and fill MODEL_SELECTION_RECORD with the selected checkpoint, validation metric, code revision, and date. The cell refuses to overwrite an existing final metrics file.

The report records the split seed, test size, group count, plate count, grouping limitations, model-selection record, operational threshold, and COCO scoring threshold. Do not use this test output to revise the model or threshold.

In [ ]:
from pathlib import Path
import json
from src.evaluate import evaluate_detector

RUN_FINAL_TEST_EVALUATION = False
FINAL_OPENVINO_CHECKPOINT = REPO_ROOT / "models/usa/detector/rtdetrv2_s/openvino"
FINAL_TEST_METRICS_PATH = REPO_ROOT / "results/metrics/rtdetr_usa_independent_test.json"

if RUN_FINAL_TEST_EVALUATION:
    if not globals().get("INDEPENDENT_TEST_READY", False):
        raise RuntimeError("Create and audit the grouped holdout split first.")
    if not MODEL_CHECKPOINT_FROZEN or not MODEL_SELECTION_RECORD.strip():
        raise RuntimeError("Record and freeze the selected model using validation only.")
    if not FINAL_OPENVINO_CHECKPOINT.is_dir():
        raise FileNotFoundError(f"OpenVINO checkpoint does not exist: {FINAL_OPENVINO_CHECKPOINT}")
    threshold_file = METRICS_DIR / "rtdetr_usa_threshold_calibration.json"
    if not threshold_file.is_file():
        raise RuntimeError("Calibrate the USA confidence threshold on validation first.")
    threshold_record = json.loads(threshold_file.read_text(encoding="utf-8"))
    if (
        threshold_record.get("checkpoint") != str(FINAL_OPENVINO_CHECKPOINT)
        or threshold_record.get("model_selection_record") != MODEL_SELECTION_RECORD
    ):
        raise RuntimeError("The saved threshold does not match the selected USA checkpoint.")
    ACTIVE_THRESHOLD = float(threshold_record["threshold"])
    if FINAL_TEST_METRICS_PATH.exists():
        raise FileExistsError(
            f"{FINAL_TEST_METRICS_PATH} already exists; the final test will not be evaluated again."
        )
    if SPLIT_SUMMARY["splits"]["test"]["images"] < MINIMUM_TEST_IMAGES:
        raise RuntimeError("The test split is smaller than the configured minimum.")

    FINAL_TEST_METRICS_PATH.parent.mkdir(parents=True, exist_ok=True)
    temporary_metrics_path = FINAL_TEST_METRICS_PATH.with_suffix(".working.json")
    if temporary_metrics_path.exists():
        temporary_metrics_path.unlink()

    final_test_metrics = evaluate_detector(
        data_dir=DATA_DIR,
        country="usa",
        split="test",
        checkpoint=FINAL_OPENVINO_CHECKPOINT,
        model_id=MODEL_ID,
        threshold=0.001,
        result_path=temporary_metrics_path,
    )
    final_test_metrics["test_protocol"] = {
        "source_dataset": SPLIT_SUMMARY["source_dataset"],
        "seed": SPLIT_SEED,
        "test_design": SPLIT_SUMMARY["test_design"],
        "test_images": SPLIT_SUMMARY["splits"]["test"]["images"],
        "test_groups": SPLIT_SUMMARY["splits"]["test"]["groups"],
        "test_plates": SPLIT_SUMMARY["splits"]["test"]["plates"],
        "grouping_level": SPLIT_SUMMARY["grouping_level"],
        "model_selection_record": MODEL_SELECTION_RECORD,
        "threshold_for_ap": 0.001,
        "operational_threshold": ACTIVE_THRESHOLD,
    }
    FINAL_TEST_METRICS_PATH.write_text(
        json.dumps(final_test_metrics, indent=2, ensure_ascii=False), encoding="utf-8"
    )
    temporary_metrics_path.unlink(missing_ok=True)
    print(json.dumps(final_test_metrics, indent=2, ensure_ascii=False))
else:
    print(
        "The final test has not been evaluated. Train and select using train/validation; "
        "freeze the model and then set RUN_FINAL_TEST_EVALUATION to True exactly once."
    )

## 24. Pre-deployment checklist

Before using the model in access decisions, confirm that:

- The USA data covers different plate designs and states, day/night conditions, glare, weather, angles, distances, small plates, and occlusions.
- Related images are grouped by vehicle, camera, session, or sequence when those identifiers exist. Inspect the saved split manifest for overlap and exact duplicates.
- The detector report includes mAP50-95, AP50, AP75, precision, recall, F1, false positives, false negatives, and small-object results.
- The confidence threshold was selected on validation according to the costs of false acceptances and false rejections.
- Robustness diagnostics include real scenes as well as synthetic image perturbations.
- Crop OCR and full-scene end-to-end metrics are reported separately, including exact-match rate, character error rate, and missed detections.
- CPU latency uses a documented hardware class and the same input size as deployment.
- Any future automated gate is assessed with a separate labeled decision dataset; uncertain readings are sent for human review.

A high detector score alone does not establish a safe access-control system. The starting checkpoint's published metrics describe its original evaluation set and are not USA SIRAP results.

## 25. Optionally publish the reviewed OpenVINO IR

Publishing is disabled by default. After reviewing and documenting the final results, set PUBLISH_IR_TO_SPACE to True. The helper requires a Colab secret named GITHUB_TOKEN with write access to the project repository. It publishes the country-specific OpenVINO files; repository automation can then synchronize them with the Hugging Face Space. This step does not publish the training dataset.

In [ ]:
PUBLISH_IR_TO_SPACE = False
if PUBLISH_IR_TO_SPACE:
    from scripts.notebook_io import publish_detector_ir
    if not COUNTRY_IR_XML.is_file():
        raise FileNotFoundError("Train and export the USA model first.")
    print(publish_detector_ir(REPO_ROOT, COUNTRY))
else:
    print("Publishing is disabled.")

## 26. Result files and reporting

No training or evaluation results are embedded in this notebook yet. The cells generate files only when their run switches are enabled and the required data is available. Do not fill in or report metric values until those runs actually finish.

- The grouped split builder writes split_summary.json, split_manifest.csv, and an audit CSV for exact duplicates with conflicting annotations.
- Optuna writes rtdetr_usa_optuna_trials.csv and rtdetr_usa_optuna_summary.json. The three-seed training stage writes rtdetr_usa_three_seed_validation.csv/.json and copies the selected run manifest to rtdetr_usa_training_manifest.json. Use validation metrics to select the checkpoint and record that decision.
- The one-time detector evaluation writes rtdetr_usa_independent_test.json. This is the held-out detector report and must not be used to modify the model or threshold.
- Threshold calibration writes rtdetr_usa_threshold_calibration.json; robustness and CPU timing write image_robustness_rtdetr_usa.csv and openvino_usa_latency.json. These are diagnostics and must be labeled with their data split and hardware conditions.
- OCR crop, direct scene-level, and SIRAP pipeline end-to-end results are stored separately; they measure OCR or detection-plus-OCR, not detector mAP.

The imported training helper also writes its training manifest. File names and metric definitions are part of the report: state the dataset/version, split sizes, number of groups and plates, checkpoint, confidence threshold, runtime, and whether a metric came from validation or the final test.
